# Black-Litterman: rebalanceo, pandemia y comparacion con Markowitz

Este notebook implementa una version autocontenida del experimento Black-Litterman para el universo F5 usado en la entrega. Replica la logica final del analisis Markowitz: benchmark top-20, universo de 601 activos, rebalanceo semestral, escenarios con/sin pandemia, splits aleatorios y ventanas moviles robustas.

## Requerimientos y orden de ejecucion

1. Ejecutar desde la raiz del repositorio: `x:/Capstone final`.
2. Mantener esta estructura esperada:

```text
Capstone final/
|-- Historical_Stocks/
|   |-- stock_return_<TICKER>.csv
|-- Entrega 2/
|   |-- Historical_Stocks_filtrado_sin_pandemia/
|   |   |-- tickers_filtrados_F5.csv
|   |   |-- Historical_Stocks_sin_pandemia/
|   |       |-- stock_return_<TICKER>.csv
|   |-- Implementación caso benchmark teórico/
|   |   |-- markowitz_vs_benchmark_rebalanceo_pandemia.ipynb
|   |   |-- ../01_markowitz_vs_benchmark/outputs/
|   |   |-- black_litterman.ipynb
|   |   |-- outputs/
|   |-- Implementación múltiples slipts/
|   |   |-- outputs_markowitz_cvxpy_splits_aleatorios/validation_windows.csv
|   |-- IMPLEMENTACIÓN OFICIAL/
|       |-- outputs_markowitz_cvxpy_variante_validacion/validation_windows.csv
```

3. Ejecutar las celdas en orden. Primero se validan rutas y dependencias; luego se corre smoke test con 10 activos; despues el experimento central `h7_f2`; al final se procesan las 70 ventanas aleatorias y las 18 ventanas robustas.

## Salidas principales

Todos los archivos se guardan en `outputs/`. La comparacion principal se hace contra `../01_markowitz_vs_benchmark/outputs/`.


## Formato de lectura del notebook

Este Jupyter queda organizado como informe reproducible:

1. Requerimientos, carpeta esperada y setup.
2. Metodologia Black-Litterman y construccion de views.
3. Smoke test de implementacion.
4. Experimento central `h7_f2` con rebalanceo.
5. Comparacion con/sin pandemia.
6. Comparacion BL vs Markowitz vs Benchmark.
7. Validacion por 70 splits aleatorios.
8. Validacion robusta con 18 ventanas moviles.
9. Conclusiones finales.

Las tablas y graficos de resultados estan insertados en el notebook, no solo guardados como archivos externos.


## Codigo base Markowitz embebido
Esta celda contiene el codigo base necesario, sin dependencias `.py` externas.

In [1]:
# Codigo base Markowitz embebido. No requiere archivo .py externo.

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Variante del modelo de Markowitz con benchmark teorico usando CVXPY.

Este script trabaja con la base:
Desarrollo modelo Markowitz/Data/Historical_Stocks_filtrado_sin_pandemia

Que calcula:
1. Retornos diarios totales, incluyendo dividendos.
2. Splits historico/futuro de maximo 9 anios: 7-2, 6-3, ..., 2-7.
3. Portafolios Markowitz para cinco perfiles de riesgo.
4. Benchmark equiponderado de empresas con mayor capitalizacion.
5. Portafolio de minima varianza.
6. Portafolio ideal de mercado, aproximado como maximo Sharpe/tangencia.
7. Frontera eficiente teorica sola.
8. Frontera eficiente con portafolios relevantes senalados.
9. Frontera eficiente teorica versus desempeno realizado en periodos futuros.
10. Validacion robusta con multiples ventanas historico/futuro por horizonte.
11. Creacion del perfil de riesgo B_teorico a partir de la volatilidad
    anualizada del benchmark y calculo del portafolio b_teorico: Markowitz
    resuelto con esa cota de riesgo propia.



Si CVXPY no esta instalado:
pip install cvxpy
"""

from __future__ import annotations

import argparse
import math
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


TRADING_DAYS_PER_YEAR = 252
DEFAULT_HORIZONS = [(7, 2), (6, 3), (5, 4), (4, 5), (3, 6), (2, 7)]
LOCAL_DEPS = Path.cwd() / ".deps_markowitz"
# Solo se usa si el usuario crea explicitamente este marcador.
# Evita que instalaciones locales incompletas en OneDrive contaminen imports.
if (LOCAL_DEPS / ".enable").exists():
    sys.path.insert(0, str(LOCAL_DEPS))


@dataclass(frozen=True)
class RiskProfile:
    key: str
    label: str
    gamma: float
    max_weight: float
    annual_vol_cap: Optional[float]
    loss_tolerance: float
    marker: str


RISK_PROFILES: Dict[str, RiskProfile] = {
    # gamma controla la penalizacion de varianza.
    # annual_vol_cap es una traduccion operacional de la tolerancia al riesgo.
    # Si la cota no es factible, el script relaja solo esa cota y deja registro.
    "muy_conservador": RiskProfile(
        "muy_conservador", "Muy conservador", 80.0, 0.05, 0.08, 0.00, "o"
    ),
    "conservador": RiskProfile(
        "conservador", "Conservador", 45.0, 0.07, 0.12, 0.05, "s"
    ),
    "neutro": RiskProfile("neutro", "Neutro", 20.0, 0.10, 0.18, 0.15, "^"),
    "arriesgado": RiskProfile(
        "arriesgado", "Arriesgado", 8.0, 0.15, 0.28, 0.30, "D"
    ),
    "muy_arriesgado": RiskProfile(
        "muy_arriesgado", "Muy arriesgado", 3.0, 0.20, 0.40, 0.40, "P"
    ),
}


SPECIAL_MARKERS = {
    "Benchmark": ("X", "#111111"),
    "b_teorico": ("h", "#D55E00"),
    "Minima varianza": ("*", "#008080"),
    "Ideal de mercado": ("v", "#B00020"),
}


def import_cvxpy():
    try:
        import cvxpy as cp
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "CVXPY no esta instalado en este entorno de Python.\n"
            "Instalalo con: pip install cvxpy\n"
            "Luego vuelve a ejecutar este script."
        ) from exc
    return cp


def import_pyplot():
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "matplotlib no esta instalado en este entorno de Python.\n"
            "Instalalo con: pip install matplotlib\n"
            "O ejecuta el script con --skip-plots para omitir graficos."
        ) from exc
    return plt


def parse_horizon(value: str) -> Tuple[int, int]:
    clean = value.strip().lower().replace(" ", "")
    for sep in (":", "-", ","):
        if sep in clean:
            left, right = clean.split(sep, 1)
            return int(left), int(right)
    raise argparse.ArgumentTypeError(
        "Cada horizonte debe tener formato historico:futuro, por ejemplo 8:2."
    )


def default_paths() -> Tuple[Path, Path]:
    script_dir = Path.cwd()
    entrega_2 = script_dir.parent
    data_root = (
        entrega_2
        / "Desarrollo modelo Markowitz"
        / "Data"
        / "Historical_Stocks_filtrado_sin_pandemia"
    )
    output_dir = script_dir / "outputs_markowitz_cvxpy_benchmark_teorico"
    return data_root, output_dir


def read_metadata(data_root: Path) -> pd.DataFrame:
    meta_path = data_root / "tickers_filtrados_F5.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"No se encontro metadata F5: {meta_path}")

    meta = pd.read_csv(meta_path)
    required = {"ticker", "sector", "industry", "marketCap"}
    missing = required.difference(meta.columns)
    if missing:
        raise ValueError(f"Faltan columnas en tickers_filtrados_F5.csv: {missing}")

    meta = meta.copy()
    meta["ticker"] = meta["ticker"].astype(str).str.upper().str.strip()
    meta["marketCap"] = pd.to_numeric(meta["marketCap"], errors="coerce")
    meta = meta.dropna(subset=["ticker", "marketCap"])
    meta = meta.drop_duplicates(subset=["ticker"], keep="first")
    meta = meta.sort_values("marketCap", ascending=False).reset_index(drop=True)
    return meta


def select_market_universe(meta: pd.DataFrame, size: int) -> List[str]:
    if size == 0:
        return meta["ticker"].tolist()
    if size <= 1:
        raise ValueError("--market-universe-size debe ser 0 para todos o mayor a 1.")
    return meta.head(size)["ticker"].tolist()


def read_stock_total_return(csv_path: Path, ticker: str) -> pd.Series:
    if not csv_path.exists():
        raise FileNotFoundError(f"No existe archivo para {ticker}: {csv_path}")

    df = pd.read_csv(csv_path, usecols=["Date", "Close", "Dividends"])
    df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
    df["Date"] = df["Date"].dt.tz_convert(None).dt.normalize()
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df["Dividends"] = pd.to_numeric(df["Dividends"], errors="coerce").fillna(0.0)
    df = df.dropna(subset=["Date", "Close"])
    df = df[df["Close"] > 0]
    df = df.sort_values("Date").drop_duplicates("Date", keep="last").set_index("Date")

    total_return = (df["Close"].diff() + df["Dividends"]) / df["Close"].shift(1)
    total_return = total_return.replace([np.inf, -np.inf], np.nan).dropna()
    total_return.name = ticker
    return total_return


def load_returns_matrix(data_root: Path, tickers: Sequence[str]) -> pd.DataFrame:
    stocks_dir = data_root / "Historical_Stocks_sin_pandemia"
    if not stocks_dir.exists():
        raise FileNotFoundError(f"No se encontro carpeta de CSV: {stocks_dir}")

    series = []
    for ticker in tickers:
        csv_path = stocks_dir / f"stock_return_{ticker}.csv"
        series.append(read_stock_total_return(csv_path, ticker))

    # Inner join para que todos los activos compartan las mismas fechas.
    returns = pd.concat(series, axis=1, join="inner").dropna(how="any")
    returns = returns.sort_index()

    if returns.shape[1] < 2:
        raise ValueError("Se requieren al menos dos activos para Markowitz.")
    return returns


def make_split(
    returns: pd.DataFrame,
    historical_years: int,
    future_years: int,
    periods_per_year: int = TRADING_DAYS_PER_YEAR,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    return make_split_window(
        returns=returns,
        historical_years=historical_years,
        future_years=future_years,
        start_idx=len(returns) - (historical_years + future_years) * periods_per_year,
        periods_per_year=periods_per_year,
    )


def make_split_window(
    returns: pd.DataFrame,
    historical_years: int,
    future_years: int,
    start_idx: int,
    periods_per_year: int = TRADING_DAYS_PER_YEAR,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    hist_days = historical_years * periods_per_year
    future_days = future_years * periods_per_year
    total_days = hist_days + future_days

    if len(returns) < total_days:
        raise ValueError(
            f"No hay suficientes observaciones comunes: se requieren {total_days}, "
            f"pero hay {len(returns)}."
        )

    if start_idx < 0 or start_idx + total_days > len(returns):
        raise ValueError(
            f"Ventana invalida: start_idx={start_idx}, total_days={total_days}, "
            f"observaciones={len(returns)}."
        )

    window = returns.iloc[start_idx : start_idx + total_days]
    historical = window.iloc[:hist_days].copy()
    future = window.iloc[hist_days:].copy()
    return historical, future


def generate_validation_starts(
    n_observations: int,
    historical_years: int,
    future_years: int,
    n_windows: int,
    step_days: int,
    periods_per_year: int = TRADING_DAYS_PER_YEAR,
) -> List[int]:
    """Genera inicios de ventanas para validar robustez temporal.

    Si n_windows > 0, se eligen ventanas espaciadas de forma uniforme entre la
    primera y la ultima ventana factible. Si n_windows == 0, se usan todas las
    ventanas separadas por step_days. La ultima ventana factible siempre queda
    incluida para conservar la validacion mas reciente.
    """
    total_days = (historical_years + future_years) * periods_per_year
    max_start = n_observations - total_days
    if max_start < 0:
        raise ValueError(
            f"La matriz comun tiene {n_observations} retornos diarios, "
            f"pero el horizonte maximo requiere {total_days}."
        )

    if max_start == 0:
        return [0]

    if n_windows > 0:
        count = min(n_windows, max_start + 1)
        starts = np.linspace(0, max_start, count)
        starts = sorted({int(round(x)) for x in starts})
    else:
        if step_days <= 0:
            raise ValueError("--validation-step-days debe ser positivo.")
        starts = list(range(0, max_start + 1, step_days))
        if starts[-1] != max_start:
            starts.append(max_start)

    return starts


def annualize_return(daily_return: float) -> float:
    return (1.0 + daily_return) ** TRADING_DAYS_PER_YEAR - 1.0


def annualize_vol(daily_vol: float) -> float:
    return daily_vol * math.sqrt(TRADING_DAYS_PER_YEAR)


def risk_free_daily(risk_free_rate_annual: float) -> float:
    return (1.0 + risk_free_rate_annual) ** (1.0 / TRADING_DAYS_PER_YEAR) - 1.0


def estimate_parameters(
    historical_returns: pd.DataFrame,
    shrinkage: float,
) -> Tuple[np.ndarray, np.ndarray]:
    mu = historical_returns.mean().values.astype(float)
    cov = historical_returns.cov().values.astype(float)
    cov = shrink_covariance(cov, shrinkage)
    cov = nearest_psd(cov)
    return mu, cov


def shrink_covariance(cov: np.ndarray, shrinkage: float) -> np.ndarray:
    if not (0.0 <= shrinkage <= 1.0):
        raise ValueError("--shrinkage debe estar entre 0 y 1.")
    diag = np.diag(np.diag(cov))
    return (1.0 - shrinkage) * cov + shrinkage * diag


def nearest_psd(matrix: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    sym = 0.5 * (matrix + matrix.T)
    eigvals, eigvecs = np.linalg.eigh(sym)
    eigvals = np.clip(eigvals, eps, None)
    psd = (eigvecs * eigvals) @ eigvecs.T
    return 0.5 * (psd + psd.T)


@dataclass
class OptimizationResult:
    name: str
    weights: np.ndarray
    status: str
    objective_value: Optional[float]
    elapsed_seconds: float
    note: str = ""


def solve_cvxpy_problem(
    problem,
    cp,
    solver_hint: Optional[str] = None,
    installed_solvers: Optional[Iterable[str]] = None,
) -> str:
    installed = set(installed_solvers) if installed_solvers is not None else set(cp.installed_solvers())
    preferred = []
    if solver_hint:
        preferred.append(solver_hint)
    preferred.extend(["CLARABEL", "OSQP", "ECOS", "SCS"])

    last_error = None
    for solver in preferred:
        if solver not in installed:
            continue
        try:
            problem.solve(solver=solver, verbose=False)
            return solver
        except Exception as exc:  # pragma: no cover - depends on local solvers.
            last_error = exc
            continue

    try:
        problem.solve(verbose=False)
        return "default"
    except Exception as exc:
        last_error = exc

    raise RuntimeError(f"No se pudo resolver el problema CVXPY: {last_error}")


def base_constraints(cp, w, n: int, max_weight: float):
    return [cp.sum(w) == 1.0, w >= 0.0, w <= max_weight]


def solve_profile_portfolio(
    cp,
    mu: np.ndarray,
    sigma: np.ndarray,
    profile: RiskProfile,
    solver: Optional[str],
) -> OptimizationResult:
    n = len(mu)
    w = cp.Variable(n)
    sigma_psd = cp.psd_wrap(sigma)
    constraints = base_constraints(cp, w, n, profile.max_weight)

    if profile.annual_vol_cap is not None:
        daily_var_cap = (profile.annual_vol_cap**2) / TRADING_DAYS_PER_YEAR
        constraints_with_cap = constraints + [cp.quad_form(w, sigma_psd) <= daily_var_cap]
    else:
        constraints_with_cap = constraints

    objective = cp.Maximize(mu @ w - 0.5 * profile.gamma * cp.quad_form(w, sigma_psd))

    start = time.perf_counter()
    problem = cp.Problem(objective, constraints_with_cap)
    used_solver = solve_cvxpy_problem(problem, cp, solver)
    elapsed = time.perf_counter() - start

    note = f"solver={used_solver}; vol_cap_activa={profile.annual_vol_cap is not None}"
    if problem.status not in {"optimal", "optimal_inaccurate"} or w.value is None:
        # Relajacion controlada: deja registro, pero conserva long-only, presupuesto y max_weight.
        start = time.perf_counter()
        problem = cp.Problem(objective, constraints)
        used_solver = solve_cvxpy_problem(problem, cp, solver)
        elapsed = time.perf_counter() - start
        note = f"solver={used_solver}; vol_cap_relajada_por_infactibilidad"

    if problem.status not in {"optimal", "optimal_inaccurate"} or w.value is None:
        raise RuntimeError(f"No se pudo resolver perfil {profile.label}: {problem.status}")

    weights = clean_weights(np.asarray(w.value).ravel())
    return OptimizationResult(
        profile.label, weights, problem.status, problem.value, elapsed, note
    )


def solve_min_variance(
    cp,
    mu: np.ndarray,
    sigma: np.ndarray,
    max_weight: float,
    solver: Optional[str],
) -> OptimizationResult:
    del mu
    n = sigma.shape[0]
    w = cp.Variable(n)
    sigma_psd = cp.psd_wrap(sigma)
    objective = cp.Minimize(cp.quad_form(w, sigma_psd))
    constraints = base_constraints(cp, w, n, max_weight)

    start = time.perf_counter()
    problem = cp.Problem(objective, constraints)
    used_solver = solve_cvxpy_problem(problem, cp, solver)
    elapsed = time.perf_counter() - start

    if problem.status not in {"optimal", "optimal_inaccurate"} or w.value is None:
        raise RuntimeError(f"No se pudo resolver minima varianza: {problem.status}")

    return OptimizationResult(
        "Minima varianza",
        clean_weights(np.asarray(w.value).ravel()),
        problem.status,
        problem.value,
        elapsed,
        f"solver={used_solver}",
    )


def solve_max_return(
    cp,
    mu: np.ndarray,
    sigma: np.ndarray,
    max_weight: float,
    solver: Optional[str],
) -> OptimizationResult:
    del sigma
    n = len(mu)
    w = cp.Variable(n)
    objective = cp.Maximize(mu @ w)
    constraints = base_constraints(cp, w, n, max_weight)

    start = time.perf_counter()
    problem = cp.Problem(objective, constraints)
    used_solver = solve_cvxpy_problem(problem, cp, solver)
    elapsed = time.perf_counter() - start

    if problem.status not in {"optimal", "optimal_inaccurate"} or w.value is None:
        raise RuntimeError(f"No se pudo resolver maximo retorno: {problem.status}")

    return OptimizationResult(
        "Maximo retorno",
        clean_weights(np.asarray(w.value).ravel()),
        problem.status,
        problem.value,
        elapsed,
        f"solver={used_solver}",
    )


@dataclass
class FrontierPoint:
    target_daily_return: float
    weights: np.ndarray
    status: str
    elapsed_seconds: float


def solve_efficient_frontier(
    cp,
    mu: np.ndarray,
    sigma: np.ndarray,
    max_weight: float,
    n_points: int,
    solver: Optional[str],
) -> List[FrontierPoint]:
    min_var = solve_min_variance(cp, mu, sigma, max_weight, solver)
    max_ret = solve_max_return(cp, mu, sigma, max_weight, solver)

    min_ret = float(mu @ min_var.weights)
    max_ret_value = float(mu @ max_ret.weights)
    if max_ret_value <= min_ret:
        targets = np.array([min_ret])
    else:
        targets = np.linspace(min_ret, max_ret_value, n_points)

    n = len(mu)
    sigma_psd = cp.psd_wrap(sigma)
    installed_solvers = set(cp.installed_solvers())
    points: List[FrontierPoint] = []
    w = cp.Variable(n)
    target_param = cp.Parameter(name="target_daily_return")
    objective = cp.Minimize(cp.quad_form(w, sigma_psd))
    constraints = base_constraints(cp, w, n, max_weight) + [mu @ w >= target_param]
    problem = cp.Problem(objective, constraints)
    used_solver: Optional[str] = None

    for target in targets:
        target_param.value = float(target)
        start = time.perf_counter()
        try:
            if used_solver is None:
                used_solver = solve_cvxpy_problem(
                    problem, cp, solver, installed_solvers=installed_solvers
                )
            elif used_solver == "default":
                problem.solve(verbose=False, warm_start=True)
            else:
                problem.solve(solver=used_solver, verbose=False, warm_start=True)
        except Exception:
            try:
                used_solver = solve_cvxpy_problem(
                    problem, cp, solver, installed_solvers=installed_solvers
                )
            except Exception as exc:
                warnings.warn(f"Frontera: target {target:.8f} fallo: {exc}")
                continue
        elapsed = time.perf_counter() - start

        if problem.status in {"optimal", "optimal_inaccurate"} and w.value is not None:
            points.append(
                FrontierPoint(
                    float(target),
                    clean_weights(np.asarray(w.value).ravel()),
                    problem.status,
                    elapsed,
                )
            )

    return points


def select_tangency_from_frontier(
    frontier: Sequence[FrontierPoint],
    historical_returns: pd.DataFrame,
    risk_free_rate_annual: float,
) -> OptimizationResult:
    if not frontier:
        raise ValueError("No hay puntos de frontera para elegir tangencia.")

    mu = historical_returns.mean().values
    sigma = nearest_psd(historical_returns.cov().values)
    rf_daily = risk_free_daily(risk_free_rate_annual)

    best_point = None
    best_sharpe = -np.inf
    for point in frontier:
        daily_mean = float(mu @ point.weights)
        daily_var = float(point.weights @ sigma @ point.weights)
        daily_vol = math.sqrt(max(daily_var, 0.0))
        sharpe = (daily_mean - rf_daily) / daily_vol if daily_vol > 0 else -np.inf
        if sharpe > best_sharpe:
            best_point = point
            best_sharpe = sharpe

    assert best_point is not None
    return OptimizationResult(
        "Ideal de mercado",
        best_point.weights,
        best_point.status,
        best_sharpe,
        best_point.elapsed_seconds,
        f"max_sharpe_sobre_frontera; rf_anual={risk_free_rate_annual}",
    )


def clean_weights(weights: np.ndarray, tolerance: float = 1e-8) -> np.ndarray:
    w = np.asarray(weights, dtype=float)
    w[np.abs(w) < tolerance] = 0.0
    w[w < 0.0] = 0.0
    total = w.sum()
    if total <= 0:
        raise ValueError("Vector de pesos invalido: suma cero.")
    return w / total


def benchmark_equal_weight(n_assets: int, universe_size: int) -> np.ndarray:
    if n_assets <= 0:
        raise ValueError("--benchmark-size debe ser positivo.")
    n = min(n_assets, universe_size)
    weights = np.zeros(universe_size)
    weights[:n] = 1.0 / n
    return weights


def create_benchmark_risk_profile(
    benchmark_weights: np.ndarray,
    historical_returns: pd.DataFrame,
    max_weight: float,
) -> Tuple[RiskProfile, float]:
    """Crea el perfil B_teorico usando la volatilidad historica del benchmark.

    A diferencia de los perfiles FinPUC discretos, B_teorico no clasifica el
    benchmark dentro de una categoria existente. Construye una categoria nueva
    cuya cota de volatilidad anual es exactamente la volatilidad anualizada del
    benchmark equiponderado. Con gamma=0, la optimizacion busca el mayor retorno
    esperado sujeto al mismo nivel de riesgo del benchmark, ubicando el punto
    teorico comparable sobre la frontera eficiente.
    """
    benchmark_returns = portfolio_daily_returns(historical_returns, benchmark_weights)
    benchmark_vol_ann = annualize_vol(float(benchmark_returns.std(ddof=1)))
    profile = RiskProfile(
        key="b_teorico",
        label="B_teorico",
        gamma=0.0,
        max_weight=max_weight,
        annual_vol_cap=benchmark_vol_ann,
        loss_tolerance=np.nan,
        marker=SPECIAL_MARKERS["b_teorico"][0],
    )
    return profile, benchmark_vol_ann


def solve_benchmark_theoretical_portfolio(
    cp,
    mu: np.ndarray,
    sigma: np.ndarray,
    benchmark_weights: np.ndarray,
    historical_returns: pd.DataFrame,
    max_weight: float,
    solver: Optional[str],
) -> Tuple[OptimizationResult, RiskProfile, float]:
    """Resuelve b_teorico: Markowitz con el perfil propio B_teorico."""
    benchmark_profile, benchmark_vol_ann = create_benchmark_risk_profile(
        benchmark_weights, historical_returns, max_weight
    )
    result = solve_profile_portfolio(cp, mu, sigma, benchmark_profile, solver)
    result = OptimizationResult(
        "b_teorico",
        result.weights,
        result.status,
        result.objective_value,
        result.elapsed_seconds,
        (
            f"perfil_benchmark={benchmark_profile.label}; "
            f"vol_benchmark_teorica={benchmark_vol_ann:.6f}; "
            f"gamma_b_teorico={benchmark_profile.gamma:.6f}; "
            f"{result.note}"
        ),
    )
    return result, benchmark_profile, benchmark_vol_ann


def portfolio_daily_returns(returns: pd.DataFrame, weights: np.ndarray) -> pd.Series:
    values = returns.values @ weights
    return pd.Series(values, index=returns.index, name="portfolio_return")


def cvar_losses(portfolio_returns: pd.Series, alpha: float = 0.95) -> float:
    losses = -portfolio_returns.dropna()
    if losses.empty:
        return np.nan
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(tail.mean()) if len(tail) else np.nan


def max_drawdown_from_returns(portfolio_returns: pd.Series) -> float:
    wealth = (1.0 + portfolio_returns).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0
    return float(drawdown.min())


def evaluate_portfolio(
    weights: np.ndarray,
    historical_returns: pd.DataFrame,
    future_returns: pd.DataFrame,
    risk_free_rate_annual: float,
    initial_capital: float,
) -> Dict[str, float]:
    hist_r = portfolio_daily_returns(historical_returns, weights)
    fut_r = portfolio_daily_returns(future_returns, weights)

    hist_mean = float(hist_r.mean())
    fut_mean = float(fut_r.mean())
    hist_vol_daily = float(hist_r.std(ddof=1))
    fut_vol_daily = float(fut_r.std(ddof=1))

    hist_ret_ann = annualize_return(hist_mean)
    fut_ret_ann = annualize_return(fut_mean)
    hist_vol_ann = annualize_vol(hist_vol_daily)
    fut_vol_ann = annualize_vol(fut_vol_daily)

    rf = risk_free_rate_annual
    hist_sharpe = (hist_ret_ann - rf) / hist_vol_ann if hist_vol_ann > 0 else np.nan
    fut_sharpe = (fut_ret_ann - rf) / fut_vol_ann if fut_vol_ann > 0 else np.nan

    future_wealth = initial_capital * float((1.0 + fut_r).cumprod().iloc[-1])

    return {
        "retorno_diario_teorico": hist_mean,
        "volatilidad_diaria_teorica": hist_vol_daily,
        "retorno_anual_teorico": hist_ret_ann,
        "volatilidad_anual_teorica": hist_vol_ann,
        "sharpe_teorico": hist_sharpe,
        "retorno_diario_real_futuro": fut_mean,
        "volatilidad_diaria_real_futuro": fut_vol_daily,
        "retorno_anual_real_futuro": fut_ret_ann,
        "volatilidad_anual_real_futuro": fut_vol_ann,
        "sharpe_real_futuro": fut_sharpe,
        "riqueza_final_futuro": future_wealth,
        "retorno_total_futuro": future_wealth / initial_capital - 1.0,
        "max_drawdown_futuro": max_drawdown_from_returns(fut_r),
        "cvar_95_diario_futuro": cvar_losses(fut_r, alpha=0.95),
    }


def make_summary_record(
    split_label: str,
    historical_years: int,
    future_years: int,
    portfolio_type: str,
    result: OptimizationResult,
    tickers: Sequence[str],
    historical_returns: pd.DataFrame,
    future_returns: pd.DataFrame,
    risk_free_rate_annual: float,
    initial_capital: float,
) -> Dict[str, object]:
    metrics = evaluate_portfolio(
        result.weights,
        historical_returns,
        future_returns,
        risk_free_rate_annual,
        initial_capital,
    )
    nonzero = int(np.sum(result.weights > 1e-6))
    max_w = float(result.weights.max())
    top_idx = np.argsort(result.weights)[::-1][:5]
    top_5 = "; ".join(
        f"{tickers[i]}={result.weights[i]:.2%}" for i in top_idx if result.weights[i] > 1e-6
    )

    record: Dict[str, object] = {
        "split": split_label,
        "historical_years": historical_years,
        "future_years": future_years,
        "portfolio": result.name,
        "portfolio_type": portfolio_type,
        "n_assets_positive": nonzero,
        "max_weight": max_w,
        "top_5_weights": top_5,
        "status": result.status,
        "objective_value": result.objective_value,
        "elapsed_seconds": result.elapsed_seconds,
        "note": result.note,
    }
    record.update(metrics)
    return record


def weights_to_records(
    split_label: str,
    result: OptimizationResult,
    tickers: Sequence[str],
    meta_by_ticker: pd.DataFrame,
) -> List[Dict[str, object]]:
    records = []
    for ticker, weight in zip(tickers, result.weights):
        if weight <= 1e-8:
            continue
        row = meta_by_ticker.loc[ticker]
        records.append(
            {
                "split": split_label,
                "portfolio": result.name,
                "ticker": ticker,
                "weight": float(weight),
                "sector": row.get("sector", ""),
                "industry": row.get("industry", ""),
                "marketCap": row.get("marketCap", np.nan),
                "nombre": row.get("nombre", ""),
            }
        )
    return records


def frontier_to_records(
    split_label: str,
    frontier: Sequence[FrontierPoint],
    historical_returns: pd.DataFrame,
    future_returns: pd.DataFrame,
    risk_free_rate_annual: float,
    initial_capital: float,
) -> List[Dict[str, object]]:
    rows = []
    for idx, point in enumerate(frontier, start=1):
        metrics = evaluate_portfolio(
            point.weights,
            historical_returns,
            future_returns,
            risk_free_rate_annual,
            initial_capital,
        )
        rows.append(
            {
                "split": split_label,
                "frontier_point": idx,
                "target_daily_return": point.target_daily_return,
                "status": point.status,
                "elapsed_seconds": point.elapsed_seconds,
                **metrics,
            }
        )
    return rows


def plot_frontier_comparison(
    output_path: Path,
    split_label: str,
    frontier_records: pd.DataFrame,
    portfolio_records: pd.DataFrame,
):
    plt = import_pyplot()
    fig, ax = plt.subplots(figsize=(11, 7))

    frontier = frontier_records.sort_values("volatilidad_anual_teorica")
    ax.plot(
        frontier["volatilidad_anual_teorica"],
        frontier["retorno_anual_teorico"],
        color="#1f77b4",
        linewidth=2.2,
        label="Frontera eficiente teorica",
    )
    ax.scatter(
        frontier["volatilidad_anual_real_futuro"],
        frontier["retorno_anual_real_futuro"],
        color="#9a9a9a",
        alpha=0.45,
        s=30,
        label="Mismos pesos evaluados en futuro real",
    )

    colors = {
        "Perfil": "#ff7f0e",
        "Benchmark": "#111111",
        "Especial": "#008080",
    }

    for _, row in portfolio_records.iterrows():
        if row["portfolio_type"] == "Perfil":
            profile = next(
                p for p in RISK_PROFILES.values() if p.label == row["portfolio"]
            )
            marker = profile.marker
            color = colors["Perfil"]
        elif row["portfolio"] == "Benchmark":
            marker, color = SPECIAL_MARKERS["Benchmark"]
        elif row["portfolio"] == "b_teorico":
            marker, color = SPECIAL_MARKERS["b_teorico"]
        elif row["portfolio"] == "Ideal de mercado":
            marker, color = SPECIAL_MARKERS["Ideal de mercado"]
        else:
            marker, color = SPECIAL_MARKERS["Minima varianza"]

        x0 = row["volatilidad_anual_teorica"]
        y0 = row["retorno_anual_teorico"]
        x1 = row["volatilidad_anual_real_futuro"]
        y1 = row["retorno_anual_real_futuro"]

        ax.scatter(x0, y0, marker=marker, s=115, color=color, edgecolor="white", zorder=4)
        ax.scatter(x1, y1, marker=marker, s=95, facecolor="none", edgecolor=color, zorder=4)
        ax.plot([x0, x1], [y0, y1], linestyle="--", linewidth=0.9, color=color, alpha=0.55)
        ax.annotate(
            row["portfolio"],
            xy=(x0, y0),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=8,
        )

    ax.set_title(f"Frontera eficiente y validacion futura - {split_label}")
    ax.set_xlabel("Volatilidad anualizada")
    ax.set_ylabel("Retorno anualizado")
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")

    # Leyenda compacta: teorico/reales y convencion de marcadores.
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="best", fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_frontier_theoretical_only(
    output_path: Path,
    split_label: str,
    frontier_records: pd.DataFrame,
):
    plt = import_pyplot()
    fig, ax = plt.subplots(figsize=(10, 6.5))

    frontier = frontier_records.sort_values("volatilidad_anual_teorica")
    ax.plot(
        frontier["volatilidad_anual_teorica"],
        frontier["retorno_anual_teorico"],
        color="#1f77b4",
        linewidth=2.4,
        label="Frontera eficiente teorica",
    )

    ax.set_title(f"Frontera eficiente teorica - {split_label}")
    ax.set_xlabel("Volatilidad anualizada teorica")
    ax.set_ylabel("Retorno anualizado teorico")
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_frontier_with_portfolios(
    output_path: Path,
    split_label: str,
    frontier_records: pd.DataFrame,
    portfolio_records: pd.DataFrame,
):
    plt = import_pyplot()
    fig, ax = plt.subplots(figsize=(11, 7))

    frontier = frontier_records.sort_values("volatilidad_anual_teorica")
    ax.plot(
        frontier["volatilidad_anual_teorica"],
        frontier["retorno_anual_teorico"],
        color="#1f77b4",
        linewidth=2.2,
        label="Frontera eficiente teorica",
    )

    for _, row in portfolio_records.iterrows():
        if row["portfolio_type"] == "Perfil":
            profile = next(
                p for p in RISK_PROFILES.values() if p.label == row["portfolio"]
            )
            marker = profile.marker
            color = "#ff7f0e"
        elif row["portfolio"] == "Benchmark":
            marker, color = SPECIAL_MARKERS["Benchmark"]
        elif row["portfolio"] == "b_teorico":
            marker, color = SPECIAL_MARKERS["b_teorico"]
        elif row["portfolio"] == "Ideal de mercado":
            marker, color = SPECIAL_MARKERS["Ideal de mercado"]
        else:
            marker, color = SPECIAL_MARKERS["Minima varianza"]

        ax.scatter(
            row["volatilidad_anual_teorica"],
            row["retorno_anual_teorico"],
            marker=marker,
            s=120,
            color=color,
            edgecolor="white",
            zorder=4,
        )
        ax.annotate(
            row["portfolio"],
            xy=(row["volatilidad_anual_teorica"], row["retorno_anual_teorico"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=8,
        )

    ax.set_title(f"Frontera eficiente con portafolios - {split_label}")
    ax.set_xlabel("Volatilidad anualizada teorica")
    ax.set_ylabel("Retorno anualizado teorico")
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_horizon_comparison(output_path: Path, summary: pd.DataFrame):
    plt = import_pyplot()
    profiles = [profile.label for profile in RISK_PROFILES.values()]
    subset = summary[summary["portfolio"].isin(profiles + ["Benchmark", "b_teorico", "Ideal de mercado", "Minima varianza"])]

    fig, ax = plt.subplots(figsize=(12, 7))
    for portfolio, group in subset.groupby("portfolio"):
        agg = (
            group.groupby("horizon", as_index=False)["sharpe_real_futuro"]
            .agg(["mean", "std"])
            .reset_index()
        )
        order = {f"h{h}_f{f}": i for i, (h, f) in enumerate(DEFAULT_HORIZONS)}
        agg["order"] = agg["horizon"].map(order).fillna(999)
        agg = agg.sort_values("order")
        ax.errorbar(
            agg["horizon"],
            agg["mean"],
            yerr=agg["std"].fillna(0.0),
            marker="o",
            linewidth=1.8,
            capsize=3,
            label=portfolio,
        )

    ax.set_title("Validacion robusta: Sharpe futuro medio por horizonte")
    ax.set_xlabel("Split historico:futuro")
    ax.set_ylabel("Sharpe futuro medio (+/- desviacion)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8, ncol=2)
    fig.autofmt_xdate(rotation=30)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def run_split(
    cp,
    split_label: str,
    horizon_label: str,
    window_id: int,
    window_start_idx: int,
    historical_years: int,
    future_years: int,
    returns: pd.DataFrame,
    meta: pd.DataFrame,
    tickers: Sequence[str],
    args,
) -> Tuple[List[Dict[str, object]], List[Dict[str, object]], List[Dict[str, object]], Dict[str, object]]:
    historical, future = make_split_window(
        returns, historical_years, future_years, window_start_idx
    )
    mu, sigma = estimate_parameters(historical, args.shrinkage)

    split_dir = args.output_dir / horizon_label / split_label
    split_dir.mkdir(parents=True, exist_ok=True)

    portfolio_results: List[Tuple[str, OptimizationResult]] = []

    for profile in RISK_PROFILES.values():
        result = solve_profile_portfolio(cp, mu, sigma, profile, args.solver)
        portfolio_results.append(("Perfil", result))

    benchmark_w = benchmark_equal_weight(args.benchmark_size, len(tickers))
    portfolio_results.append(
        (
            "Benchmark",
            OptimizationResult(
                "Benchmark",
                benchmark_w,
                "closed_form",
                None,
                0.0,
                f"equiponderado_top_{min(args.benchmark_size, len(tickers))}_marketCap",
            ),
        )
    )

    b_teorico_result, benchmark_profile, benchmark_vol_ann = solve_benchmark_theoretical_portfolio(
        cp,
        mu,
        sigma,
        benchmark_w,
        historical,
        args.frontier_max_weight,
        args.solver,
    )
    portfolio_results.append(("Benchmark teorico", b_teorico_result))

    min_var_result = solve_min_variance(
        cp, mu, sigma, args.frontier_max_weight, args.solver
    )
    portfolio_results.append(("Especial", min_var_result))

    frontier = solve_efficient_frontier(
        cp,
        mu,
        sigma,
        args.frontier_max_weight,
        args.frontier_points,
        args.solver,
    )
    tangency_result = select_tangency_from_frontier(
        frontier, historical, args.risk_free_rate
    )
    portfolio_results.append(("Especial", tangency_result))

    summary_rows = [
        make_summary_record(
            split_label,
            historical_years,
            future_years,
            portfolio_type,
            result,
            tickers,
            historical,
            future,
            args.risk_free_rate,
            args.initial_capital,
        )
        for portfolio_type, result in portfolio_results
    ]
    for row in summary_rows:
        row.update(
            {
                "horizon": horizon_label,
                "window_id": window_id,
                "window_start_idx": window_start_idx,
                "benchmark_risk_profile": benchmark_profile.label,
                "benchmark_volatility_ann": benchmark_vol_ann,
                "historical_start": historical.index.min(),
                "historical_end": historical.index.max(),
                "future_start": future.index.min(),
                "future_end": future.index.max(),
            }
        )

    meta_by_ticker = meta.set_index("ticker")
    weight_rows: List[Dict[str, object]] = []
    for _, result in portfolio_results:
        weight_rows.extend(weights_to_records(split_label, result, tickers, meta_by_ticker))
    for row in weight_rows:
        row.update(
            {
                "horizon": horizon_label,
                "window_id": window_id,
                "benchmark_risk_profile": benchmark_profile.label,
                "benchmark_volatility_ann": benchmark_vol_ann,
            }
        )

    frontier_rows = frontier_to_records(
        split_label, frontier, historical, future, args.risk_free_rate, args.initial_capital
    )
    for row in frontier_rows:
        row.update(
            {
                "horizon": horizon_label,
                "window_id": window_id,
                "benchmark_risk_profile": benchmark_profile.label,
                "benchmark_volatility_ann": benchmark_vol_ann,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    frontier_df = pd.DataFrame(frontier_rows)

    summary_df.to_csv(split_dir / "portfolios_summary.csv", index=False)
    pd.DataFrame(weight_rows).to_csv(split_dir / "weights.csv", index=False)
    frontier_df.to_csv(split_dir / "frontier_points.csv", index=False)

    if not args.skip_plots and not frontier_df.empty:
        plot_frontier_theoretical_only(
            split_dir / f"frontera_teorica_sola_{split_label}.png",
            split_label,
            frontier_df,
        )
        plot_frontier_with_portfolios(
            split_dir / f"frontera_teorica_portafolios_{split_label}.png",
            split_label,
            frontier_df,
            summary_df,
        )
        plot_frontier_comparison(
            split_dir / f"frontera_teorica_vs_real_{split_label}.png",
            split_label,
            frontier_df,
            summary_df,
        )

    window_info = {
        "split": split_label,
        "horizon": horizon_label,
        "window_id": window_id,
        "window_start_idx": window_start_idx,
        "benchmark_risk_profile": benchmark_profile.label,
        "benchmark_volatility_ann": benchmark_vol_ann,
        "historical_years": historical_years,
        "future_years": future_years,
        "historical_start": historical.index.min(),
        "historical_end": historical.index.max(),
        "future_start": future.index.min(),
        "future_end": future.index.max(),
        "historical_observations": len(historical),
        "future_observations": len(future),
        "n_assets": len(tickers),
    }
    return summary_rows, weight_rows, frontier_rows, window_info


def build_arg_parser() -> argparse.ArgumentParser:
    data_root, output_dir = default_paths()
    parser = argparse.ArgumentParser(
        description="Variante robusta de Markowitz FinPUC con CVXPY y validacion movil."
    )
    parser.add_argument("--data-root", type=Path, default=data_root)
    parser.add_argument("--output-dir", type=Path, default=output_dir)
    parser.add_argument(
        "--market-universe-size",
        type=int,
        default=601, # Se define la cantidad de activos con los que trabajar 
        help="0 usa todos los activos F5; N usa los top N por capitalizacion.",
    )
    parser.add_argument("--benchmark-size", type=int, default=20)
    parser.add_argument("--frontier-points", type=int, default=30)
    parser.add_argument("--frontier-max-weight", type=float, default=0.20)
    parser.add_argument("--risk-free-rate", type=float, default=0.02)
    parser.add_argument("--initial-capital", type=float, default=1000.0)
    parser.add_argument("--shrinkage", type=float, default=0.20)
    parser.add_argument(
        "--horizons",
        type=parse_horizon,
        nargs="+",
        default=DEFAULT_HORIZONS,
        help="Splits historico:futuro, maximo sugerido 9 anios. Ejemplo: --horizons 7:2 6:3",
    )
    parser.add_argument(
        "--validation-windows",
        type=int,
        default=3,
        help="Numero de ventanas por horizonte. 0 usa todas segun --validation-step-days.",
    )
    parser.add_argument(
        "--validation-step-days",
        type=int,
        default=63,
        help="Separacion entre ventanas cuando --validation-windows es 0.",
    )
    parser.add_argument(
        "--solver",
        type=str,
        default=None,
        help="Solver CVXPY opcional, por ejemplo CLARABEL, OSQP o SCS.",
    )
    parser.add_argument("--skip-plots", action="store_true")
    return parser


def main() -> int:
    parser = build_arg_parser()
    args = parser.parse_args()
    args.data_root = args.data_root.resolve()
    args.output_dir = args.output_dir.resolve()
    args.horizons = list(dict.fromkeys(args.horizons))

    cp = import_cvxpy()

    args.output_dir.mkdir(parents=True, exist_ok=True)

    meta = read_metadata(args.data_root)
    tickers = select_market_universe(meta, args.market_universe_size)
    returns = load_returns_matrix(args.data_root, tickers)

    max_required_days = max((h + f) * TRADING_DAYS_PER_YEAR for h, f in args.horizons)
    if len(returns) < max_required_days:
        raise ValueError(
            f"La matriz comun tiene {len(returns)} retornos diarios, "
            f"pero el horizonte maximo requiere {max_required_days}."
        )

    all_summary: List[Dict[str, object]] = []
    all_weights: List[Dict[str, object]] = []
    all_frontier: List[Dict[str, object]] = []
    all_windows: List[Dict[str, object]] = []

    print("Markowitz CVXPY FinPUC - variante con validacion movil")
    print(f"Data root: {args.data_root}")
    print(f"Output dir: {args.output_dir}")
    print(f"Activos usados: {len(tickers)}")
    print(f"Observaciones comunes: {len(returns)}")
    print(f"Ventanas de validacion por horizonte: {args.validation_windows}")
    print(f"Solvers CVXPY instalados: {', '.join(cp.installed_solvers())}")

    for historical_years, future_years in args.horizons:
        horizon_label = f"h{historical_years}_f{future_years}"
        starts = generate_validation_starts(
            n_observations=len(returns),
            historical_years=historical_years,
            future_years=future_years,
            n_windows=args.validation_windows,
            step_days=args.validation_step_days,
        )
        print(f"\nResolviendo horizonte {horizon_label} con {len(starts)} ventanas...")

        horizon_summary_rows = []
        for window_id, start_idx in enumerate(starts, start=1):
            split_label = f"{horizon_label}_w{window_id:02d}"
            print(f"  Ventana {window_id}/{len(starts)}: {split_label}")
            summary_rows, weight_rows, frontier_rows, window_info = run_split(
                cp,
                split_label,
                horizon_label,
                window_id,
                start_idx,
                historical_years,
                future_years,
                returns,
                meta,
                tickers,
                args,
            )
            all_summary.extend(summary_rows)
            all_weights.extend(weight_rows)
            all_frontier.extend(frontier_rows)
            all_windows.append(window_info)
            horizon_summary_rows.extend(summary_rows)

        horizon_summary = pd.DataFrame(horizon_summary_rows)
        quick = (
            horizon_summary.groupby("portfolio", as_index=False)
            .agg(
                sharpe_futuro_promedio=("sharpe_real_futuro", "mean"),
                sharpe_futuro_std=("sharpe_real_futuro", "std"),
                retorno_futuro_promedio=("retorno_anual_real_futuro", "mean"),
                volatilidad_futura_promedio=("volatilidad_anual_real_futuro", "mean"),
            )
            .sort_values("sharpe_futuro_promedio", ascending=False)
        )
        print(quick.to_string(index=False))

    summary = pd.DataFrame(all_summary)
    weights = pd.DataFrame(all_weights)
    frontier = pd.DataFrame(all_frontier)
    windows = pd.DataFrame(all_windows)
    robustness = (
        summary.groupby(["horizon", "portfolio", "portfolio_type"], as_index=False)
        .agg(
            n_windows=("window_id", "nunique"),
            sharpe_futuro_mean=("sharpe_real_futuro", "mean"),
            sharpe_futuro_std=("sharpe_real_futuro", "std"),
            sharpe_futuro_min=("sharpe_real_futuro", "min"),
            sharpe_futuro_max=("sharpe_real_futuro", "max"),
            retorno_futuro_mean=("retorno_anual_real_futuro", "mean"),
            retorno_futuro_std=("retorno_anual_real_futuro", "std"),
            volatilidad_futura_mean=("volatilidad_anual_real_futuro", "mean"),
            max_drawdown_futuro_mean=("max_drawdown_futuro", "mean"),
            cvar_95_diario_futuro_mean=("cvar_95_diario_futuro", "mean"),
        )
        .sort_values(["horizon", "sharpe_futuro_mean"], ascending=[True, False])
    )

    summary.to_csv(args.output_dir / "portfolios_summary_all_windows.csv", index=False)
    weights.to_csv(args.output_dir / "weights_all_windows.csv", index=False)
    frontier.to_csv(args.output_dir / "frontier_points_all_windows.csv", index=False)
    windows.to_csv(args.output_dir / "validation_windows.csv", index=False)
    robustness.to_csv(args.output_dir / "validation_robustness_summary.csv", index=False)

    if not args.skip_plots and not summary.empty:
        plot_horizon_comparison(
            args.output_dir / "validacion_robusta_sharpe_por_horizonte.png", summary
        )

    print("\nListo.")
    print(f"Resumen ventanas: {args.output_dir / 'portfolios_summary_all_windows.csv'}")
    print(f"Robustez:         {args.output_dir / 'validation_robustness_summary.csv'}")
    print(f"Pesos:            {args.output_dir / 'weights_all_windows.csv'}")
    print(f"Frontera:         {args.output_dir / 'frontier_points_all_windows.csv'}")
    return 0


# CLI entry point disabled: this code is embedded in the notebook.


In [2]:
from __future__ import annotations

import math
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        entrega_2 = candidate / "Entrega 2"
        if (entrega_2 / "Historical_Stocks_filtrado_sin_pandemia").exists():
            return candidate
        if candidate.name == "Entrega 2" and (candidate / "Historical_Stocks_filtrado_sin_pandemia").exists():
            return candidate.parent
    raise FileNotFoundError("No se pudo ubicar la raiz del proyecto 'Capstone final'. Ejecuta el notebook desde una carpeta dentro del proyecto.")

REPO_ROOT = find_project_root(Path.cwd())
CASE_ROOT = REPO_ROOT / "Entrega 2" / "Implementaci\u00f3n caso benchmark te\u00f3rico"
NOTEBOOK_DIR = CASE_ROOT / "02_black_litterman"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
MARKOWITZ_OUTPUT_DIR = CASE_ROOT / "01_markowitz_vs_benchmark" / "outputs"
RANDOM_SPLITS_DIR = REPO_ROOT / "Entrega 2" / "Implementaci\u00f3n m\u00faltiples slipts" / "outputs_markowitz_cvxpy_splits_aleatorios"
ROBUST_WINDOWS_DIR = REPO_ROOT / "Entrega 2" / "IMPLEMENTACI\u00d3N OFICIAL" / "outputs_markowitz_cvxpy_variante_validacion"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cp = import_cvxpy()

SCENARIOS = {
    "sin_pandemia": {
        "label": "Sin pandemia",
        "data_root": REPO_ROOT / "Entrega 2" / "Historical_Stocks_filtrado_sin_pandemia",
        "stocks_dir": "Historical_Stocks_sin_pandemia",
        "metadata_root": REPO_ROOT / "Entrega 2" / "Historical_Stocks_filtrado_sin_pandemia",
    },
    "con_pandemia": {
        "label": "Con pandemia",
        "data_root": REPO_ROOT / "Historical_Stocks",
        "stocks_dir": "",
        "metadata_root": REPO_ROOT / "Entrega 2" / "Historical_Stocks_filtrado_sin_pandemia",
    },
}

MARKET_SIZE = int(os.getenv("BL_MARKET_SIZE", "601"))
BENCHMARK_SIZE = 20
HISTORICAL_YEARS = 7
FUTURE_YEARS = 2
REBALANCE_DAYS = 126
DEFAULT_FRONTIER_POINTS = int(os.getenv("BL_FRONTIER_POINTS", "12"))
DEFAULT_MAX_WEIGHT = 0.20
DEFAULT_RISK_FREE_RATE = 0.02
DEFAULT_SHRINKAGE = 0.20
DEFAULT_TAU = 0.05
DEFAULT_INITIAL_CAPITAL = 1000.0
DEFAULT_SOLVER = os.getenv("BL_SOLVER") or None

MOMENTUM_LOOKBACK = 252
MOMENTUM_TOP_BOTTOM = 20
MOMENTUM_CONFIDENCE = 0.50
MOMENTUM_TOP20_6M_LOOKBACK = 126
MOMENTUM_TOP20_6M_MARKET_CAP_SIZE = 20
MOMENTUM_TOP20_6M_LONG_SHORT = 10
MOMENTUM_TOP20_BOTTOM20_LOOKBACK = 252
MOMENTUM_TOP20_BOTTOM20_MARKET_CAP_SIZE = 40
MOMENTUM_TOP20_BOTTOM20_LONG_SHORT = 20

UNEMPLOYMENT_RATE_ASSUMED = 0.04
UNEMPLOYMENT_NEUTRAL = 0.05
MACRO_UNEMPLOYMENT_BETA = 1.0
UNEMPLOYMENT_CONFIDENCE = 0.35

BL_VIEW_TYPES = ("momentum", "desempleo", "momentum_top20_6m", "momentum_top20_bottom20_1y")
MARKETCAP_MOMENTUM_VIEW_TYPES = ("momentum_top20_6m", "momentum_top20_bottom20_1y")
BL_VIEW_LABELS = {
    "momentum": "BL Momentum",
    "desempleo": "BL Desempleo",
    "momentum_top20_6m": "BL Momentum Top20 6M",
    "momentum_top20_bottom20_1y": "BL Momentum Top20-Bottom20 1Y",
}
RUN_CENTRAL_ANALYSIS = os.getenv("BL_RUN_CENTRAL_ANALYSIS", os.getenv("BL_RUN_FULL_ANALYSIS", "1")) == "1"
RUN_VALIDATION_ANALYSIS = os.getenv("BL_RUN_VALIDATION_ANALYSIS", "0") == "1"
BL_FORCE_RECOMPUTE = os.getenv("BL_FORCE_RECOMPUTE", "0") == "1"
RUN_FULL_ANALYSIS = RUN_CENTRAL_ANALYSIS  # Compatibilidad con celdas antiguas.
VALIDATION_PORTFOLIOS = ("Neutro", "Minima varianza", "Benchmark")

required_paths = [
    MARKOWITZ_OUTPUT_DIR / "rebalance_summary_h7_f2_601activos.csv",
    RANDOM_SPLITS_DIR / "validation_windows.csv",
    ROBUST_WINDOWS_DIR / "validation_windows.csv",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan rutas requeridas:\n" + "\n".join(missing))

print(f"[SETUP] repo={REPO_ROOT}")
print(f"[SETUP] output={OUTPUT_DIR}")
print(f"[SETUP] market_size={MARKET_SIZE}, benchmark_size={BENCHMARK_SIZE}, frontier_points={DEFAULT_FRONTIER_POINTS}")
print(f"[SETUP] scenarios={list(SCENARIOS)}")
print(f"[SETUP] views={BL_VIEW_TYPES}")
print(f"[SETUP] run_central={RUN_CENTRAL_ANALYSIS}, run_validation={RUN_VALIDATION_ANALYSIS}, force_recompute={BL_FORCE_RECOMPUTE}")


[SETUP] repo=X:\Capstone final
[SETUP] output=X:\Capstone final\Entrega 2\Implementación caso benchmark teórico\02_black_litterman\outputs
[SETUP] market_size=601, benchmark_size=20, frontier_points=12
[SETUP] scenarios=['sin_pandemia', 'con_pandemia']
[SETUP] views=('momentum', 'desempleo', 'momentum_top20_6m', 'momentum_top20_bottom20_1y')
[SETUP] run_central=True, run_validation=False, force_recompute=True


## Metodologia Black-Litterman

El prior de equilibrio de mercado se calcula como:

`pi = delta * Sigma * w_market`

donde `w_market` son pesos por capitalizacion de mercado desde la metadata F5 y:

`delta = (retorno_mercado_anual - rf) / varianza_mercado_anual`

Con las views `P`, `Q` y `Omega`, el posterior se estima como:

`mu_bl = pi + tau*Sigma*P.T*(P*tau*Sigma*P.T + Omega)^-1*(Q - P*pi)`

La covarianza usada para optimizar es:

`Sigma_bl = nearest_psd(Sigma + posterior_covariance)`

El experimento conserva restricciones long-only, suma de pesos igual a 1 y peso maximo por activo. Los perfiles de riesgo se resuelven con la misma infraestructura CVXPY del notebook Markowitz previo.


In [3]:
@dataclass
class ScenarioDataset:
    key: str
    label: str
    returns: pd.DataFrame
    meta: pd.DataFrame
    tickers: List[str]


def load_scenario_dataset(scenario_key: str, market_size: int = MARKET_SIZE) -> ScenarioDataset:
    cfg = SCENARIOS[scenario_key]
    meta = read_metadata(cfg["metadata_root"])
    requested_tickers = select_market_universe(meta, market_size)
    stocks_root = cfg["data_root"] / cfg["stocks_dir"] if cfg["stocks_dir"] else cfg["data_root"]

    series = []
    loaded_tickers = []
    for ticker in requested_tickers:
        csv_path = stocks_root / f"stock_return_{ticker}.csv"
        if not csv_path.exists():
            continue
        series.append(read_stock_total_return(csv_path, ticker))
        loaded_tickers.append(ticker)

    if len(series) < 2:
        raise ValueError(f"{scenario_key}: se requieren al menos dos activos cargados.")

    returns = pd.concat(series, axis=1, join="inner").sort_index().dropna(how="any")
    if returns.empty:
        raise ValueError(f"{scenario_key}: matriz de retornos vacia.")

    meta_loaded = meta[meta["ticker"].isin(loaded_tickers)].copy()
    meta_loaded["ticker"] = pd.Categorical(meta_loaded["ticker"], categories=loaded_tickers, ordered=True)
    meta_loaded = meta_loaded.sort_values("ticker").reset_index(drop=True)

    print(f"[LOAD] {scenario_key}: {len(loaded_tickers)} activos, {len(returns)} observaciones, {returns.index.min().date()} a {returns.index.max().date()}")
    return ScenarioDataset(scenario_key, cfg["label"], returns, meta_loaded, loaded_tickers)


def market_cap_weights(meta: pd.DataFrame, tickers: Sequence[str]) -> np.ndarray:
    m = meta.set_index("ticker").reindex(tickers)
    caps = pd.to_numeric(m["marketCap"], errors="coerce").fillna(0.0).clip(lower=0.0).to_numpy(dtype=float)
    if caps.sum() <= 0:
        return np.ones(len(tickers)) / len(tickers)
    return caps / caps.sum()


def portfolio_type_for(name: str) -> str:
    if name in [p.label for p in RISK_PROFILES.values()]:
        return "Perfil"
    if name == "Benchmark":
        return "Benchmark"
    return "Especial"


def infer_window_by_dates(returns: pd.DataFrame, row: pd.Series) -> Tuple[pd.DataFrame, pd.DataFrame]:
    start_idx = int(row["window_start_idx"])
    h = int(row["historical_years"])
    f = int(row["future_years"])
    return make_split_window(returns, h, f, start_idx=start_idx, periods_per_year=TRADING_DAYS_PER_YEAR)


def make_rebalance_segments(returns: pd.DataFrame, historical_years: int, future_years: int, rebalance_days: int) -> List[Dict[str, object]]:
    base_train, base_future = make_split(returns, historical_years, future_years, periods_per_year=TRADING_DAYS_PER_YEAR)
    hist_days = historical_years * TRADING_DAYS_PER_YEAR
    segments = []
    future_start_pos = returns.index.get_loc(base_future.index[0])

    for segment_id, local_start in enumerate(range(0, len(base_future), rebalance_days), start=1):
        local_end = min(local_start + rebalance_days, len(base_future))
        test_segment = base_future.iloc[local_start:local_end]
        global_start = future_start_pos + local_start
        train_start = max(0, global_start - hist_days)
        train_segment = returns.iloc[train_start:global_start]
        if len(train_segment) < max(252, hist_days // 2):
            train_segment = base_train
        segments.append(
            {
                "segment_id": segment_id,
                "segment_start_idx": int(local_start),
                "segment_end_idx": int(local_end),
                "segment_start_date": str(test_segment.index[0].date()),
                "segment_end_date": str(test_segment.index[-1].date()),
                "train": train_segment,
                "test": test_segment,
            }
        )
    return segments


def aggregate_rebalanced_returns(rows_by_portfolio: Dict[str, List[pd.Series]]) -> List[Dict[str, object]]:
    rows = []
    for portfolio, pieces in rows_by_portfolio.items():
        returns = pd.concat(pieces).sort_index()
        total_return = float((1.0 + returns).prod() - 1.0)
        years = len(returns) / TRADING_DAYS_PER_YEAR
        ann_return = float((1.0 + total_return) ** (1.0 / years) - 1.0) if years > 0 else np.nan
        vol_daily = float(returns.std(ddof=1))
        vol_ann = annualize_vol(vol_daily)
        sharpe = (ann_return - DEFAULT_RISK_FREE_RATE) / vol_ann if vol_ann > 0 else np.nan
        rows.append(
            {
                "portfolio": portfolio,
                "n_segments": len(pieces),
                "retorno_total_rebalanceado": total_return,
                "retorno_anual_rebalanceado": ann_return,
                "volatilidad_diaria_rebalanceada": vol_daily,
                "volatilidad_anual_rebalanceada": vol_ann,
                "max_drawdown_rebalanceado": max_drawdown_from_returns(returns),
                "cvar_95_diario_rebalanceado": cvar_losses(returns, alpha=0.95),
                "sharpe_rebalanceado": sharpe,
            }
        )
    return rows


## Construccion de views `P`, `Q` y `Omega`

Se implementan dos variantes separadas:

- **BL Momentum**: view relativa long top-20 momentum y short bottom-20 momentum, con lookback de 252 dias habiles.
- **BL Desempleo**: view macro asumida sin datos externos. La tasa configurable es `UNEMPLOYMENT_RATE_ASSUMED = 0.04` y se compara contra neutral `0.05`. Con desempleo bajo se favorecen sectores ciclicos; con desempleo alto se favorecen defensivos.


In [4]:
CYCLICAL_SECTORS = {
    "Technology",
    "Consumer Cyclical",
    "Industrials",
    "Financial Services",
    "Basic Materials",
    "Real Estate",
    "Energy",
}
DEFENSIVE_SECTORS = {
    "Healthcare",
    "Consumer Defensive",
    "Utilities",
    "Communication Services",
}


def omega_from_view(P: np.ndarray, sigma: np.ndarray, tau: float, confidence: float) -> np.ndarray:
    base = P @ (tau * sigma) @ P.T
    omega_value = float(base[0, 0]) / max(confidence, 1e-8)
    omega_value = max(omega_value, 1e-12)
    return np.array([[omega_value]], dtype=float)


def build_momentum_view(train_returns: pd.DataFrame, sigma: np.ndarray, tickers: Sequence[str], tau: float = DEFAULT_TAU) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    lookback = min(MOMENTUM_LOOKBACK, len(train_returns))
    recent = train_returns.iloc[-lookback:]
    n_assets = len(tickers)
    k = min(MOMENTUM_TOP_BOTTOM, max(1, n_assets // 2))
    momentum = (1.0 + recent).prod(axis=0) - 1.0
    top = momentum.sort_values(ascending=False).head(k).index.tolist()
    bottom = momentum.sort_values(ascending=True).head(k).index.tolist()

    P = np.zeros((1, n_assets), dtype=float)
    for ticker in top:
        P[0, tickers.index(ticker)] = 1.0 / k
    for ticker in bottom:
        P[0, tickers.index(ticker)] = -1.0 / k

    q_value = float(recent[top].mean(axis=1).mean() - recent[bottom].mean(axis=1).mean())
    Q = np.array([q_value], dtype=float)
    Omega = omega_from_view(P, sigma, tau, MOMENTUM_CONFIDENCE)
    record = {
        "view_type": "momentum",
        "view_name": "top_bottom_momentum",
        "lookback_days": lookback,
        "n_long": len(top),
        "n_short": len(bottom),
        "q_daily": q_value,
        "omega": float(Omega[0, 0]),
        "confidence": MOMENTUM_CONFIDENCE,
        "long_side": ";".join(top[:20]),
        "short_side": ";".join(bottom[:20]),
        "fallback": "",
    }
    return P, Q, Omega, record


def build_marketcap_momentum_view(
    train_returns: pd.DataFrame,
    sigma: np.ndarray,
    tickers: Sequence[str],
    meta: pd.DataFrame,
    view_type: str,
    view_name: str,
    lookback_days: int,
    market_cap_size: int,
    long_short_size: int,
    tau: float = DEFAULT_TAU,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    lookback = min(lookback_days, len(train_returns))
    recent = train_returns.iloc[-lookback:]
    n_assets = len(tickers)

    meta_idx = meta.set_index("ticker").reindex(tickers).copy()
    meta_idx["marketCap"] = pd.to_numeric(meta_idx["marketCap"], errors="coerce").fillna(0.0)
    top_market = meta_idx.sort_values("marketCap", ascending=False).head(market_cap_size).index.tolist()
    top_market = [ticker for ticker in top_market if ticker in recent.columns]
    if len(top_market) < 2:
        top_market = list(tickers[: min(len(tickers), market_cap_size)])
        fallback = "ticker_order_fallback"
    else:
        fallback = "" if len(top_market) >= market_cap_size else f"fewer_than_{market_cap_size}_available"

    k = min(long_short_size, max(1, len(top_market) // 2))
    momentum = (1.0 + recent[top_market]).prod(axis=0) - 1.0
    top = momentum.sort_values(ascending=False).head(k).index.tolist()
    bottom = momentum.sort_values(ascending=True).head(k).index.tolist()

    P = np.zeros((1, n_assets), dtype=float)
    for ticker in top:
        P[0, tickers.index(ticker)] = 1.0 / k
    for ticker in bottom:
        P[0, tickers.index(ticker)] = -1.0 / k

    q_value = float(recent[top].mean(axis=1).mean() - recent[bottom].mean(axis=1).mean())
    Q = np.array([q_value], dtype=float)
    Omega = omega_from_view(P, sigma, tau, MOMENTUM_CONFIDENCE)
    record = {
        "view_type": view_type,
        "view_name": view_name,
        "lookback_days": lookback,
        "configured_lookback_days": lookback_days,
        "marketcap_universe_size": len(top_market),
        "configured_marketcap_size": market_cap_size,
        "configured_long_short_size": long_short_size,
        "n_long": len(top),
        "n_short": len(bottom),
        "q_daily": q_value,
        "omega": float(Omega[0, 0]),
        "confidence": MOMENTUM_CONFIDENCE,
        "long_side": ";".join(top[:long_short_size]),
        "short_side": ";".join(bottom[:long_short_size]),
        "marketcap_universe": ";".join(top_market[:market_cap_size]),
        "fallback": fallback,
    }
    return P, Q, Omega, record


def build_top20_marketcap_momentum_6m_view(
    train_returns: pd.DataFrame,
    sigma: np.ndarray,
    tickers: Sequence[str],
    meta: pd.DataFrame,
    tau: float = DEFAULT_TAU,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    return build_marketcap_momentum_view(
        train_returns,
        sigma,
        tickers,
        meta,
        view_type="momentum_top20_6m",
        view_name="top20_marketcap_momentum_6m",
        lookback_days=MOMENTUM_TOP20_6M_LOOKBACK,
        market_cap_size=MOMENTUM_TOP20_6M_MARKET_CAP_SIZE,
        long_short_size=MOMENTUM_TOP20_6M_LONG_SHORT,
        tau=tau,
    )


def build_marketcap_momentum_1y_20x20_view(
    train_returns: pd.DataFrame,
    sigma: np.ndarray,
    tickers: Sequence[str],
    meta: pd.DataFrame,
    tau: float = DEFAULT_TAU,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    return build_marketcap_momentum_view(
        train_returns,
        sigma,
        tickers,
        meta,
        view_type="momentum_top20_bottom20_1y",
        view_name="marketcap40_momentum_1y_20x20",
        lookback_days=MOMENTUM_TOP20_BOTTOM20_LOOKBACK,
        market_cap_size=MOMENTUM_TOP20_BOTTOM20_MARKET_CAP_SIZE,
        long_short_size=MOMENTUM_TOP20_BOTTOM20_LONG_SHORT,
        tau=tau,
    )


def build_unemployment_view(train_returns: pd.DataFrame, sigma: np.ndarray, tickers: Sequence[str], meta: pd.DataFrame, tau: float = DEFAULT_TAU) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    n_assets = len(tickers)
    meta_idx = meta.set_index("ticker").reindex(tickers)
    sectors = meta_idx["sector"].fillna("").astype(str)

    cyc_idx = [i for i, s in enumerate(sectors) if s in CYCLICAL_SECTORS]
    def_idx = [i for i, s in enumerate(sectors) if s in DEFENSIVE_SECTORS]
    fallback = ""
    if not cyc_idx or not def_idx:
        vols = train_returns.std(ddof=1).sort_values()
        half = max(1, n_assets // 2)
        def_names = set(vols.head(half).index)
        cyc_names = set(vols.tail(half).index)
        cyc_idx = [i for i, t in enumerate(tickers) if t in cyc_names]
        def_idx = [i for i, t in enumerate(tickers) if t in def_names]
        fallback = "volatility_split"

    signal = UNEMPLOYMENT_NEUTRAL - UNEMPLOYMENT_RATE_ASSUMED
    direction = 1.0 if signal >= 0 else -1.0
    q_value = abs(signal) * MACRO_UNEMPLOYMENT_BETA / TRADING_DAYS_PER_YEAR

    P = np.zeros((1, n_assets), dtype=float)
    for idx in cyc_idx:
        P[0, idx] = direction / len(cyc_idx)
    for idx in def_idx:
        P[0, idx] = -direction / len(def_idx)

    Q = np.array([q_value], dtype=float)
    Omega = omega_from_view(P, sigma, tau, UNEMPLOYMENT_CONFIDENCE)
    long_idx = cyc_idx if direction > 0 else def_idx
    short_idx = def_idx if direction > 0 else cyc_idx
    record = {
        "view_type": "desempleo",
        "view_name": "macro_desempleo_asumido",
        "lookback_days": len(train_returns),
        "n_long": len(long_idx),
        "n_short": len(short_idx),
        "q_daily": q_value,
        "q_formula_signed": signal * MACRO_UNEMPLOYMENT_BETA / TRADING_DAYS_PER_YEAR,
        "omega": float(Omega[0, 0]),
        "confidence": UNEMPLOYMENT_CONFIDENCE,
        "unemployment_rate_assumed": UNEMPLOYMENT_RATE_ASSUMED,
        "unemployment_neutral": UNEMPLOYMENT_NEUTRAL,
        "long_side": ";".join([tickers[i] for i in long_idx[:10]]),
        "short_side": ";".join([tickers[i] for i in short_idx[:10]]),
        "fallback": fallback,
    }
    return P, Q, Omega, record


def build_view(view_type: str, train_returns: pd.DataFrame, sigma: np.ndarray, tickers: Sequence[str], meta: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, object]]:
    if view_type == "momentum":
        return build_momentum_view(train_returns, sigma, tickers)
    if view_type == "desempleo":
        return build_unemployment_view(train_returns, sigma, tickers, meta)
    if view_type == "momentum_top20_6m":
        return build_top20_marketcap_momentum_6m_view(train_returns, sigma, tickers, meta)
    if view_type == "momentum_top20_bottom20_1y":
        return build_marketcap_momentum_1y_20x20_view(train_returns, sigma, tickers, meta)
    raise ValueError(f"view_type no soportado: {view_type}")


## Posterior Black-Litterman y optimizacion

Cada rebalanceo reestima `Sigma`, calcula el prior por capitalizacion de mercado y actualiza el retorno esperado con la view activa. Con `mu_bl` y `Sigma_bl` se resuelven los mismos portafolios que en Markowitz: cinco perfiles de riesgo, benchmark equiponderado top-20, `b_teorico`, minima varianza e ideal de mercado sobre la frontera eficiente BL.


In [5]:
def market_implied_prior(train_returns: pd.DataFrame, sigma: np.ndarray, tickers: Sequence[str], meta: pd.DataFrame, rf: float = DEFAULT_RISK_FREE_RATE) -> Tuple[np.ndarray, Dict[str, float]]:
    w_market = market_cap_weights(meta, tickers)
    market_returns = portfolio_daily_returns(train_returns, w_market)
    market_ret_ann = annualize_return(float(market_returns.mean()))
    market_var_ann = float(market_returns.var(ddof=1) * TRADING_DAYS_PER_YEAR)
    delta = (market_ret_ann - rf) / market_var_ann if market_var_ann > 0 else np.nan
    if not np.isfinite(delta):
        delta = 2.5
    pi = np.asarray(delta * sigma @ w_market, dtype=float)
    stats = {
        "delta": float(delta),
        "market_return_annual": float(market_ret_ann),
        "market_variance_annual": float(market_var_ann),
    }
    return pi, stats


def black_litterman_posterior(pi: np.ndarray, sigma: np.ndarray, P: np.ndarray, Q: np.ndarray, Omega: np.ndarray, tau: float = DEFAULT_TAU) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    sigma = nearest_psd(np.asarray(sigma, dtype=float))
    pi = np.asarray(pi, dtype=float).reshape(-1)
    Q = np.asarray(Q, dtype=float).reshape(-1)
    tau_sigma = tau * sigma
    middle = nearest_psd(P @ tau_sigma @ P.T + Omega)
    innovation = Q - P @ pi
    adjustment = tau_sigma @ P.T @ np.linalg.solve(middle, innovation)
    posterior_covariance = tau_sigma - tau_sigma @ P.T @ np.linalg.solve(middle, P @ tau_sigma)
    mu_bl = np.asarray(pi + adjustment, dtype=float).reshape(-1)
    sigma_bl = nearest_psd(sigma + posterior_covariance)
    return mu_bl, sigma_bl, posterior_covariance


def build_bl_model(train_returns: pd.DataFrame, tickers: Sequence[str], meta: pd.DataFrame, view_type: str) -> Tuple[np.ndarray, np.ndarray, Dict[str, object]]:
    _, sigma = estimate_parameters(train_returns, DEFAULT_SHRINKAGE)
    pi, prior_stats = market_implied_prior(train_returns, sigma, tickers, meta)
    P, Q, Omega, view_record = build_view(view_type, train_returns, sigma, tickers, meta)
    mu_bl, sigma_bl, posterior_cov = black_litterman_posterior(pi, sigma, P, Q, Omega, DEFAULT_TAU)
    view_record.update(
        {
            "tau": DEFAULT_TAU,
            "shrinkage": DEFAULT_SHRINKAGE,
            "rf": DEFAULT_RISK_FREE_RATE,
            "delta": prior_stats["delta"],
            "market_return_annual": prior_stats["market_return_annual"],
            "market_variance_annual": prior_stats["market_variance_annual"],
            "mu_bl_min": float(np.nanmin(mu_bl)),
            "mu_bl_max": float(np.nanmax(mu_bl)),
            "mu_bl_mean": float(np.nanmean(mu_bl)),
            "posterior_cov_trace": float(np.trace(posterior_cov)),
        }
    )
    if not np.isfinite(mu_bl).all() or not np.isfinite(sigma_bl).all():
        raise ValueError("Posterior Black-Litterman contiene NaN o infinito.")
    return mu_bl, sigma_bl, view_record


def select_tangency_bl(frontier, mu: np.ndarray, sigma: np.ndarray, risk_free_rate_annual: float = DEFAULT_RISK_FREE_RATE) -> OptimizationResult:
    if not frontier:
        raise ValueError("Frontera vacia: no se puede seleccionar tangencia.")
    rf_daily = risk_free_daily(risk_free_rate_annual)
    best = None
    best_sharpe = -np.inf
    for point in frontier:
        daily_mean = float(mu @ point.weights)
        daily_vol = math.sqrt(max(float(point.weights @ sigma @ point.weights), 0.0))
        sharpe = (daily_mean - rf_daily) / daily_vol if daily_vol > 0 else -np.inf
        if sharpe > best_sharpe:
            best = point
            best_sharpe = sharpe
    return OptimizationResult(
        "Ideal de mercado",
        best.weights,
        best.status,
        best_sharpe,
        best.elapsed_seconds,
        f"max_sharpe_sobre_frontera_bl; rf_anual={risk_free_rate_annual}",
    )


def solve_bl_snapshot(train_returns: pd.DataFrame, test_returns: pd.DataFrame, tickers: Sequence[str], meta: pd.DataFrame, view_type: str, include_full_set: bool = True):
    mu_bl, sigma_bl, view_record = build_bl_model(train_returns, tickers, meta, view_type)
    n_assets = len(tickers)
    benchmark_weights = benchmark_equal_weight(min(BENCHMARK_SIZE, n_assets), n_assets)

    results: List[Tuple[str, OptimizationResult]] = []
    if include_full_set:
        for profile in RISK_PROFILES.values():
            results.append(("Perfil", solve_profile_portfolio(cp, mu_bl, sigma_bl, profile, DEFAULT_SOLVER)))
        b_result, _, _ = solve_benchmark_theoretical_portfolio(
            cp, mu_bl, sigma_bl, benchmark_weights, train_returns, DEFAULT_MAX_WEIGHT, DEFAULT_SOLVER
        )
        results.append(("Especial", b_result))

    min_var = solve_min_variance(cp, mu_bl, sigma_bl, DEFAULT_MAX_WEIGHT, DEFAULT_SOLVER)
    benchmark = OptimizationResult("Benchmark", benchmark_weights, "closed_form", None, 0.0, f"top_{min(BENCHMARK_SIZE, n_assets)}_equal_weight")

    if include_full_set:
        frontier = solve_efficient_frontier(cp, mu_bl, sigma_bl, DEFAULT_MAX_WEIGHT, DEFAULT_FRONTIER_POINTS, DEFAULT_SOLVER)
        ideal = select_tangency_bl(frontier, mu_bl, sigma_bl)
        results.extend([("Especial", min_var), ("Benchmark", benchmark), ("Especial", ideal)])
    else:
        frontier = []
        neutral = solve_profile_portfolio(cp, mu_bl, sigma_bl, RISK_PROFILES["neutro"], DEFAULT_SOLVER)
        results.extend([("Perfil", neutral), ("Especial", min_var), ("Benchmark", benchmark)])

    return mu_bl, sigma_bl, view_record, results, frontier


def frontier_to_records_bl(split_label: str, scenario_key: str, scenario_label: str, view_type: str, segment: Dict[str, object], frontier, train_returns: pd.DataFrame, test_returns: pd.DataFrame) -> List[Dict[str, object]]:
    rows = []
    for idx, point in enumerate(frontier, start=1):
        metrics = evaluate_portfolio(point.weights, train_returns, test_returns, DEFAULT_RISK_FREE_RATE, DEFAULT_INITIAL_CAPITAL)
        rows.append(
            {
                "scenario": scenario_key,
                "scenario_label": scenario_label,
                "view_type": view_type,
                "split": split_label,
                "segment_id": segment["segment_id"],
                "frontier_point": idx,
                "target_daily_return": point.target_daily_return,
                "status": point.status,
                "elapsed_seconds": point.elapsed_seconds,
                **metrics,
            }
        )
    return rows


## Smoke test: 10 activos

Antes del calculo completo se prueba una muestra pequena para validar carga de escenarios, construccion de views, posterior BL y restricciones de pesos. Esta celda debe terminar sin NaN, con pesos long-only y suma igual a 1.


In [6]:
smoke_results = []
for scenario_key in SCENARIOS:
    dataset = load_scenario_dataset(scenario_key, market_size=10)
    train, test = make_split(dataset.returns, HISTORICAL_YEARS, FUTURE_YEARS, periods_per_year=TRADING_DAYS_PER_YEAR)
    for view_type in BL_VIEW_TYPES:
        mu_bl, sigma_bl, view_record, results, _ = solve_bl_snapshot(
            train,
            test.iloc[:min(REBALANCE_DAYS, len(test))],
            dataset.tickers,
            dataset.meta,
            view_type,
            include_full_set=False,
        )
        for _, result in results:
            weights = result.weights
            assert np.isfinite(mu_bl).all()
            assert np.isfinite(sigma_bl).all()
            assert np.isclose(weights.sum(), 1.0, atol=1e-6)
            assert (weights >= -1e-8).all()
            smoke_results.append(
                {
                    "scenario": scenario_key,
                    "view_type": view_type,
                    "portfolio": result.name,
                    "weights_sum": float(weights.sum()),
                    "min_weight": float(weights.min()),
                    "max_weight": float(weights.max()),
                    "q_daily": view_record["q_daily"],
                    "omega": view_record["omega"],
                }
            )

smoke_df = pd.DataFrame(smoke_results)
display(smoke_df)
print("[OK] smoke test Black-Litterman completado")


[LOAD] sin_pandemia: 10 activos, 2717 observaciones, 2012-05-21 a 2026-03-13


[LOAD] con_pandemia: 10 activos, 3473 observaciones, 2012-05-21 a 2026-03-13


,scenario,view_type,portfolio,weights_sum,min_weight,max_weight,q_daily,omega
0,sin_pandemia,momentum,Neutro,1.0,1.000000e-01,0.1,0.002503,0.000025
1,sin_pandemia,momentum,Minima varianza,1.0,1.646085e-07,0.2,0.002503,0.000025
2,sin_pandemia,momentum,Benchmark,1.0,1.000000e-01,0.1,0.002503,0.000025
3,sin_pandemia,desempleo,Neutro,1.0,1.000000e-01,0.1,0.000040,0.000081
4,sin_pandemia,desempleo,Minima varianza,1.0,1.700612e-07,0.2,0.000040,0.000081
5,sin_pandemia,desempleo,Benchmark,1.0,1.000000e-01,0.1,0.000040,0.000081
6,sin_pandemia,momentum_top20_6m,Neutro,1.0,1.000000e-01,0.1,0.003791,0.000025
7,sin_pandemia,momentum_top20_6m,Minima varianza,1.0,1.646085e-07,0.2,0.003791,0.000025
8,sin_pandemia,momentum_top20_6m,Benchmark,1.0,1.000000e-01,0.1,0.003791,0.000025
9,sin_pandemia,momentum_top20_bottom20_1y,Neutro,1.0,1.000000e-01,0.1,0.002503,0.000025


[OK] smoke test Black-Litterman completado


## Experimento central `h7_f2` con rebalanceo semestral

Para cada escenario (`sin_pandemia`, `con_pandemia`) y cada variante de view (`momentum`, `desempleo`) se recalibra Black-Litterman al inicio de cada semestre fuera de muestra. El test futuro de 2 anos se divide en segmentos de 126 dias habiles.


In [7]:
def run_bl_rebalance_scenario(dataset: ScenarioDataset, view_types: Sequence[str] = BL_VIEW_TYPES) -> Dict[str, pd.DataFrame]:
    split_label = f"{dataset.key}_h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_rebalance_{len(dataset.tickers)}"
    segments = make_rebalance_segments(dataset.returns, HISTORICAL_YEARS, FUTURE_YEARS, REBALANCE_DAYS)

    rebalance_summary_rows = []
    segment_rows = []
    segment_summary_rows = []
    frontier_rows = []
    view_rows = []

    for view_type in view_types:
        print(f"[CENTRAL] {dataset.key} / {view_type}: {len(segments)} segmentos")
        returns_by_portfolio: Dict[str, List[pd.Series]] = {}
        types_by_portfolio: Dict[str, str] = {}

        for segment in segments:
            train_segment = segment["train"]
            test_segment = segment["test"]
            mu_bl, sigma_bl, view_record, results, frontier = solve_bl_snapshot(
                train_segment,
                test_segment,
                dataset.tickers,
                dataset.meta,
                view_type,
                include_full_set=True,
            )
            view_record.update(
                {
                    "scenario": dataset.key,
                    "scenario_label": dataset.label,
                    "split": split_label,
                    "segment_id": segment["segment_id"],
                    "segment_start_date": segment["segment_start_date"],
                    "segment_end_date": segment["segment_end_date"],
                    "n_assets": len(dataset.tickers),
                }
            )
            view_rows.append(view_record)

            for portfolio_type, result in results:
                fut_returns = portfolio_daily_returns(test_segment, result.weights)
                returns_by_portfolio.setdefault(result.name, []).append(fut_returns)
                types_by_portfolio[result.name] = portfolio_type

                segment_rows.append(
                    {
                        "scenario": dataset.key,
                        "scenario_label": dataset.label,
                        "view_type": view_type,
                        "split": split_label,
                        "segment_id": segment["segment_id"],
                        "segment_start_idx": segment["segment_start_idx"],
                        "segment_end_idx": segment["segment_end_idx"],
                        "segment_start_date": segment["segment_start_date"],
                        "segment_end_date": segment["segment_end_date"],
                        "portfolio_type": portfolio_type,
                        "portfolio": result.name,
                        "status": result.status,
                        "segment_retorno_total": float((1.0 + fut_returns).prod() - 1.0),
                        "segment_volatilidad_diaria": float(fut_returns.std(ddof=1)),
                    }
                )

                summary = make_summary_record(
                    split_label,
                    HISTORICAL_YEARS,
                    FUTURE_YEARS,
                    portfolio_type,
                    result,
                    dataset.tickers,
                    train_segment,
                    test_segment,
                    DEFAULT_RISK_FREE_RATE,
                    DEFAULT_INITIAL_CAPITAL,
                )
                summary.update(
                    {
                        "scenario": dataset.key,
                        "scenario_label": dataset.label,
                        "view_type": view_type,
                        "segment_id": segment["segment_id"],
                        "segment_start_date": segment["segment_start_date"],
                        "segment_end_date": segment["segment_end_date"],
                    }
                )
                segment_summary_rows.append(summary)

            frontier_rows.extend(
                frontier_to_records_bl(split_label, dataset.key, dataset.label, view_type, segment, frontier, train_segment, test_segment)
            )

        for row in aggregate_rebalanced_returns(returns_by_portfolio):
            row.update(
                {
                    "scenario": dataset.key,
                    "scenario_label": dataset.label,
                    "view_type": view_type,
                    "split": split_label,
                    "portfolio_type": types_by_portfolio.get(row["portfolio"], portfolio_type_for(row["portfolio"])),
                }
            )
            rebalance_summary_rows.append(row)

    return {
        "rebalance": pd.DataFrame(rebalance_summary_rows),
        "segments": pd.DataFrame(segment_rows),
        "segment_summary": pd.DataFrame(segment_summary_rows),
        "frontier": pd.DataFrame(frontier_rows),
        "views": pd.DataFrame(view_rows),
    }


if RUN_CENTRAL_ANALYSIS:
    output_label = f"h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos"
    cache_paths = {
        "rebalance": OUTPUT_DIR / f"bl_rebalance_summary_{output_label}.csv",
        "segments": OUTPUT_DIR / f"bl_rebalance_segments_{output_label}.csv",
        "segment_summary": OUTPUT_DIR / f"bl_segment_summary_{output_label}.csv",
        "frontier": OUTPUT_DIR / f"bl_frontier_segments_{output_label}.csv",
        "views": OUTPUT_DIR / f"bl_views_{output_label}.csv",
    }

    def cache_has_expected_views(path: Path) -> bool:
        if not path.exists():
            return False
        try:
            cached = pd.read_csv(path, usecols=["view_type"])
        except Exception:
            return False
        return set(BL_VIEW_TYPES).issubset(set(cached["view_type"].dropna().unique()))

    use_cache = (
        not BL_FORCE_RECOMPUTE
        and all(path.exists() for path in cache_paths.values())
        and cache_has_expected_views(cache_paths["rebalance"])
    )

    if use_cache:
        print("[CACHE] Cargando outputs BL existentes; usa BL_FORCE_RECOMPUTE=1 para recalcular CVXPY.")
        bl_rebalance_df = pd.read_csv(cache_paths["rebalance"])
        bl_segments_df = pd.read_csv(cache_paths["segments"])
        bl_segment_summary_df = pd.read_csv(cache_paths["segment_summary"])
        bl_frontier_df = pd.read_csv(cache_paths["frontier"])
        bl_views_df = pd.read_csv(cache_paths["views"])
    else:
        central_outputs = []
        for scenario_key in ["sin_pandemia", "con_pandemia"]:
            dataset = load_scenario_dataset(scenario_key, MARKET_SIZE)
            central_outputs.append(run_bl_rebalance_scenario(dataset))

        bl_rebalance_df = pd.concat([x["rebalance"] for x in central_outputs], ignore_index=True)
        bl_segments_df = pd.concat([x["segments"] for x in central_outputs], ignore_index=True)
        bl_segment_summary_df = pd.concat([x["segment_summary"] for x in central_outputs], ignore_index=True)
        bl_frontier_df = pd.concat([x["frontier"] for x in central_outputs], ignore_index=True)
        bl_views_df = pd.concat([x["views"] for x in central_outputs], ignore_index=True)

        bl_rebalance_df.to_csv(cache_paths["rebalance"], index=False)
        bl_segments_df.to_csv(cache_paths["segments"], index=False)
        bl_segment_summary_df.to_csv(cache_paths["segment_summary"], index=False)
        bl_frontier_df.to_csv(cache_paths["frontier"], index=False)
        bl_views_df.to_csv(cache_paths["views"], index=False)
        print("[OK] Experimento central guardado")

    display(bl_rebalance_df.sort_values("sharpe_rebalanceado", ascending=False).head(12))
else:
    print("[SKIP] RUN_CENTRAL_ANALYSIS=0")


[LOAD] sin_pandemia: 601 activos, 2489 observaciones, 2013-03-08 a 2026-01-30
[CENTRAL] sin_pandemia / momentum: 4 segmentos


[CENTRAL] sin_pandemia / desempleo: 4 segmentos


[CENTRAL] sin_pandemia / momentum_top20_6m: 4 segmentos


[CENTRAL] sin_pandemia / momentum_top20_bottom20_1y: 4 segmentos


[LOAD] con_pandemia: 601 activos, 3245 observaciones, 2013-03-08 a 2026-01-30
[CENTRAL] con_pandemia / momentum: 4 segmentos


[CENTRAL] con_pandemia / desempleo: 4 segmentos


[CENTRAL] con_pandemia / momentum_top20_6m: 4 segmentos


[CENTRAL] con_pandemia / momentum_top20_bottom20_1y: 4 segmentos


[OK] Experimento central guardado


,portfolio,n_segments,retorno_total_rebalanceado,retorno_anual_rebalanceado,volatilidad_diaria_rebalanceada,volatilidad_anual_rebalanceada,max_drawdown_rebalanceado,cvar_95_diario_rebalanceado,sharpe_rebalanceado,scenario,scenario_label,view_type,split,portfolio_type
20,Neutro,4,0.722636,0.312492,0.007491,0.118921,-0.128718,0.015886,2.459544,sin_pandemia,Sin pandemia,momentum_top20_6m,sin_pandemia_h7_f2_rebalance_601,Perfil
11,Neutro,4,0.630909,0.277070,0.006641,0.105416,-0.103851,0.014058,2.438621,sin_pandemia,Sin pandemia,desempleo,sin_pandemia_h7_f2_rebalance_601,Perfil
0,Muy conservador,4,0.640050,0.280644,0.006894,0.109445,-0.101281,0.014506,2.381504,sin_pandemia,Sin pandemia,momentum,sin_pandemia_h7_f2_rebalance_601,Perfil
56,Neutro,4,0.694848,0.301863,0.007473,0.118625,-0.116782,0.015805,2.376090,con_pandemia,Con pandemia,momentum_top20_6m,con_pandemia_h7_f2_rebalance_601,Perfil
19,Conservador,4,0.621771,0.273488,0.006767,0.107426,-0.103917,0.014307,2.359649,sin_pandemia,Sin pandemia,momentum_top20_6m,sin_pandemia_h7_f2_rebalance_601,Perfil
48,Arriesgado,4,0.718662,0.310977,0.007782,0.123541,-0.133222,0.017117,2.355318,con_pandemia,Con pandemia,desempleo,con_pandemia_h7_f2_rebalance_601,Perfil
28,Conservador,4,0.630836,0.277042,0.006896,0.109466,-0.106955,0.014846,2.348146,sin_pandemia,Sin pandemia,momentum_top20_bottom20_1y,sin_pandemia_h7_f2_rebalance_601,Perfil
1,Conservador,4,0.684786,0.297993,0.007465,0.118506,-0.111846,0.015304,2.345821,sin_pandemia,Sin pandemia,momentum,sin_pandemia_h7_f2_rebalance_601,Perfil
27,Muy conservador,4,0.606202,0.267360,0.006778,0.107596,-0.097840,0.014403,2.298966,sin_pandemia,Sin pandemia,momentum_top20_bottom20_1y,sin_pandemia_h7_f2_rebalance_601,Perfil
10,Conservador,4,0.594729,0.262826,0.006662,0.105750,-0.096173,0.013944,2.296222,sin_pandemia,Sin pandemia,desempleo,sin_pandemia_h7_f2_rebalance_601,Perfil


### Resultado 1 - Experimento central h7_f2 con rebalanceo

Esta tabla queda dentro del notebook como resultado visible del experimento central. Resume los mejores portafolios Black-Litterman, ya agregados sobre los 4 rebalanceos semestrales.

| Escenario | View | Portafolio | Retorno total | Vol anual | Drawdown | Sharpe |
| --- | --- | --- | --- | --- | --- | --- |
| Sin pandemia | desempleo | Neutro | 63.1% | 10.5% | -10.4% | 2.44 |
| Sin pandemia | momentum | Muy conservador | 64.0% | 10.9% | -10.1% | 2.38 |
| Con pandemia | desempleo | Arriesgado | 71.9% | 12.4% | -13.3% | 2.36 |
| Sin pandemia | momentum | Conservador | 68.5% | 11.9% | -11.2% | 2.35 |
| Sin pandemia | desempleo | Conservador | 59.5% | 10.6% | -9.6% | 2.30 |
| Sin pandemia | desempleo | Muy conservador | 59.2% | 10.7% | -9.4% | 2.26 |
| Sin pandemia | desempleo | Arriesgado | 75.0% | 13.6% | -15.4% | 2.23 |
| Sin pandemia | desempleo | Minima varianza | 57.0% | 10.9% | -10.0% | 2.14 |
| Sin pandemia | momentum | Minima varianza | 56.9% | 10.9% | -10.0% | 2.14 |
| Con pandemia | desempleo | Neutro | 56.4% | 10.8% | -10.4% | 2.13 |

**Lectura del resultado**

- Sin pandemia / desempleo: lidera **Neutro** con Sharpe 2.44, retorno total 63.1% y drawdown -10.4%.
- Sin pandemia / momentum: lidera **Muy conservador** con Sharpe 2.38, retorno total 64.0% y drawdown -10.1%.
- Con pandemia / desempleo: lidera **Arriesgado** con Sharpe 2.36, retorno total 71.9% y drawdown -13.3%.
- Con pandemia / momentum: lidera **Conservador** con Sharpe 2.06, retorno total 56.4% y drawdown -10.3%.

La comparacion se debe leer con Sharpe rebalanceado porque los pesos se recalibran cada semestre. El retorno total aislado puede favorecer perfiles agresivos, pero el Sharpe penaliza la volatilidad adicional.


## Lectura de resultados centrales

La tabla anterior ordena por Sharpe rebalanceado fuera de muestra. Aqui se resume el mejor portafolio por escenario y por tipo de view, separando el efecto de pandemia y el efecto de la matriz de views.


In [8]:
if RUN_CENTRAL_ANALYSIS:
    best_central = (
        bl_rebalance_df.sort_values("sharpe_rebalanceado", ascending=False)
        .groupby(["scenario", "scenario_label", "view_type"], as_index=False)
        .head(1)
        [["scenario_label", "view_type", "portfolio", "retorno_total_rebalanceado", "volatilidad_anual_rebalanceada", "max_drawdown_rebalanceado", "sharpe_rebalanceado"]]
    )
    display(best_central)

    pandemic_comparison = (
        bl_rebalance_df.pivot_table(
            index=["view_type", "portfolio", "portfolio_type"],
            columns="scenario",
            values=["retorno_total_rebalanceado", "retorno_anual_rebalanceado", "volatilidad_anual_rebalanceada", "max_drawdown_rebalanceado", "cvar_95_diario_rebalanceado", "sharpe_rebalanceado"],
            aggfunc="first",
        )
    )
    pandemic_comparison.columns = [f"{metric}_{scenario}" for metric, scenario in pandemic_comparison.columns]
    pandemic_comparison = pandemic_comparison.reset_index()
    if "sharpe_rebalanceado_con_pandemia" in pandemic_comparison and "sharpe_rebalanceado_sin_pandemia" in pandemic_comparison:
        pandemic_comparison["delta_sharpe_con_menos_sin"] = pandemic_comparison["sharpe_rebalanceado_con_pandemia"] - pandemic_comparison["sharpe_rebalanceado_sin_pandemia"]
        pandemic_comparison["delta_retorno_total_con_menos_sin"] = pandemic_comparison["retorno_total_rebalanceado_con_pandemia"] - pandemic_comparison["retorno_total_rebalanceado_sin_pandemia"]
        pandemic_comparison["delta_drawdown_con_menos_sin"] = pandemic_comparison["max_drawdown_rebalanceado_con_pandemia"] - pandemic_comparison["max_drawdown_rebalanceado_sin_pandemia"]

    pandemic_path = OUTPUT_DIR / f"comparison_bl_sin_vs_con_pandemia_h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos.csv"
    pandemic_comparison.to_csv(pandemic_path, index=False)
    display(pandemic_comparison.sort_values("delta_sharpe_con_menos_sin").head(10))
    print(f"[OK] {pandemic_path.name}")


,scenario_label,view_type,portfolio,retorno_total_rebalanceado,volatilidad_anual_rebalanceada,max_drawdown_rebalanceado,sharpe_rebalanceado
20,Sin pandemia,momentum_top20_6m,Neutro,0.722636,0.118921,-0.128718,2.459544
11,Sin pandemia,desempleo,Neutro,0.630909,0.105416,-0.103851,2.438621
0,Sin pandemia,momentum,Muy conservador,0.640050,0.109445,-0.101281,2.381504
56,Con pandemia,momentum_top20_6m,Neutro,0.694848,0.118625,-0.116782,2.376090
48,Con pandemia,desempleo,Arriesgado,0.718662,0.123541,-0.133222,2.355318
28,Sin pandemia,momentum_top20_bottom20_1y,Conservador,0.630836,0.109466,-0.106955,2.348146
65,Con pandemia,momentum_top20_bottom20_1y,Neutro,0.656337,0.121418,-0.127863,2.198921
37,Con pandemia,momentum,Conservador,0.563570,0.111616,-0.103267,2.064471


,view_type,portfolio,portfolio_type,cvar_95_diario_rebalanceado_con_pandemia,cvar_95_diario_rebalanceado_sin_pandemia,max_drawdown_rebalanceado_con_pandemia,max_drawdown_rebalanceado_sin_pandemia,retorno_anual_rebalanceado_con_pandemia,retorno_anual_rebalanceado_sin_pandemia,retorno_total_rebalanceado_con_pandemia,retorno_total_rebalanceado_sin_pandemia,sharpe_rebalanceado_con_pandemia,sharpe_rebalanceado_sin_pandemia,volatilidad_anual_rebalanceada_con_pandemia,volatilidad_anual_rebalanceada_sin_pandemia,delta_sharpe_con_menos_sin,delta_retorno_total_con_menos_sin,delta_drawdown_con_menos_sin
15,momentum,Muy conservador,Perfil,0.014503,0.014506,-0.098502,-0.101281,0.240730,0.280644,0.539411,0.640050,2.008761,2.381504,0.109884,0.109445,-0.372743,-0.100638,0.002779
7,desempleo,Neutro,Perfil,0.014640,0.014058,-0.103743,-0.103851,0.250679,0.277070,0.564198,0.630909,2.131193,2.438621,0.108239,0.105416,-0.307428,-0.066711,0.000108
6,desempleo,Muy conservador,Perfil,0.014615,0.014098,-0.096895,-0.093773,0.237158,0.261809,0.530560,0.592161,1.962617,2.263907,0.110647,0.106810,-0.301290,-0.061601,-0.003122
2,desempleo,Conservador,Perfil,0.014597,0.013944,-0.099014,-0.096173,0.240413,0.262826,0.538625,0.594729,2.005471,2.296222,0.109906,0.105750,-0.290751,-0.056104,-0.002841
11,momentum,Conservador,Perfil,0.014667,0.015304,-0.103267,-0.111846,0.250428,0.297993,0.563570,0.684786,2.064471,2.345821,0.111616,0.118506,-0.281350,-0.121216,0.008579
33,momentum_top20_bottom20_1y,Muy conservador,Perfil,0.014676,0.014403,-0.096779,-0.097840,0.242709,0.267360,0.544325,0.606202,2.019300,2.298966,0.110290,0.107596,-0.279667,-0.061877,0.001060
20,momentum_top20_6m,Conservador,Perfil,0.014646,0.014307,-0.097396,-0.103917,0.250522,0.273488,0.563805,0.621771,2.106264,2.359649,0.109446,0.107426,-0.253385,-0.057966,0.006521
24,momentum_top20_6m,Muy conservador,Perfil,0.014552,0.014246,-0.095956,-0.097456,0.242026,0.263256,0.542628,0.595816,2.019796,2.266552,0.109925,0.107324,-0.246756,-0.053188,0.001501
29,momentum_top20_bottom20_1y,Conservador,Perfil,0.014888,0.014846,-0.099788,-0.106955,0.254100,0.277042,0.572766,0.630836,2.116267,2.348146,0.110619,0.109466,-0.231879,-0.058070,0.007167
31,momentum_top20_bottom20_1y,Minima varianza,Especial,0.014700,0.014576,-0.097600,-0.099529,0.234761,0.252683,0.524634,0.569215,1.929804,2.135240,0.111286,0.108973,-0.205436,-0.044581,0.001929


[OK] comparison_bl_sin_vs_con_pandemia_h7_f2_601activos.csv


### Resultado 2 - Sensibilidad con y sin pandemia

La siguiente tabla compara el mismo portafolio y view contra las dos bases de datos. Un delta Sharpe positivo indica que incluir los anos 2020-2022 mejora el desempeno fuera de muestra del caso evaluado.

| View | Portafolio | Sharpe sin | Sharpe con | Delta Sharpe | Delta drawdown |
| --- | --- | --- | --- | --- | --- |
| momentum | b_teorico | 1.02 | 1.49 | 0.47 | 13.6% |
| momentum | Muy arriesgado | 0.95 | 1.14 | 0.19 | 1.2% |
| desempleo | Arriesgado | 2.23 | 2.36 | 0.12 | 2.1% |
| momentum | Arriesgado | 1.31 | 1.34 | 0.03 | 2.9% |
| momentum | Neutro | 1.83 | 1.85 | 0.01 | 0.7% |
| desempleo | Benchmark | 2.05 | 2.05 | -0.00 | -0.0% |
| momentum | Benchmark | 2.05 | 2.05 | -0.00 | -0.0% |
| momentum | Ideal de mercado | 1.31 | 1.27 | -0.04 | 0.5% |

![Comparacion pandemia](outputs/comparison_bl_sin_vs_con_pandemia.png)

**Lectura del resultado:** la pandemia no debe tratarse como un simple ruido a eliminar. En algunos perfiles aporta informacion de estres util; en otros deteriora el Sharpe por mayor volatilidad y drawdown. La columna `delta_sharpe_con_menos_sin` es la decision cuantitativa para cada portafolio.


## Comparacion Black-Litterman vs Markowitz vs Benchmark

Se leen los CSV finales del modelo Markowitz con rebalanceo y pandemia. La comparacion se hace por escenario y portafolio, agregando la dimension `view_type` para distinguir BL Momentum y BL Desempleo.


In [9]:
def read_markowitz_rebalance() -> pd.DataFrame:
    path = MARKOWITZ_OUTPUT_DIR / f"rebalance_summary_h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos.csv"
    df = pd.read_csv(path)
    if "scenario" not in df.columns:
        first = df.columns[0]
        if first.startswith("Unnamed"):
            df = df.rename(columns={first: "scenario"})
        else:
            df = df.rename(columns={first: "scenario"})
    return df


if RUN_CENTRAL_ANALYSIS:
    markowitz_df = read_markowitz_rebalance()
    markowitz_keep = markowitz_df[
        [
            "scenario",
            "scenario_label",
            "portfolio",
            "retorno_total_rebalanceado",
            "retorno_anual_rebalanceado",
            "volatilidad_anual_rebalanceada",
            "max_drawdown_rebalanceado",
            "sharpe_rebalanceado",
        ]
    ].copy()
    markowitz_keep = markowitz_keep.rename(
        columns={
            "retorno_total_rebalanceado": "retorno_total_markowitz",
            "retorno_anual_rebalanceado": "retorno_anual_markowitz",
            "volatilidad_anual_rebalanceada": "volatilidad_anual_markowitz",
            "max_drawdown_rebalanceado": "max_drawdown_markowitz",
            "sharpe_rebalanceado": "sharpe_markowitz",
        }
    )
    bl_keep = bl_rebalance_df[
        [
            "scenario",
            "scenario_label",
            "view_type",
            "portfolio",
            "portfolio_type",
            "retorno_total_rebalanceado",
            "retorno_anual_rebalanceado",
            "volatilidad_anual_rebalanceada",
            "max_drawdown_rebalanceado",
            "sharpe_rebalanceado",
        ]
    ].rename(
        columns={
            "retorno_total_rebalanceado": "retorno_total_bl",
            "retorno_anual_rebalanceado": "retorno_anual_bl",
            "volatilidad_anual_rebalanceada": "volatilidad_anual_bl",
            "max_drawdown_rebalanceado": "max_drawdown_bl",
            "sharpe_rebalanceado": "sharpe_bl",
        }
    )
    comparison_bl_vs_markowitz = bl_keep.merge(
        markowitz_keep,
        on=["scenario", "scenario_label", "portfolio"],
        how="left",
    )
    comparison_bl_vs_markowitz["delta_sharpe_bl_menos_markowitz"] = comparison_bl_vs_markowitz["sharpe_bl"] - comparison_bl_vs_markowitz["sharpe_markowitz"]
    comparison_bl_vs_markowitz["delta_retorno_total_bl_menos_markowitz"] = comparison_bl_vs_markowitz["retorno_total_bl"] - comparison_bl_vs_markowitz["retorno_total_markowitz"]
    comparison_bl_vs_markowitz["delta_retorno_anual_bl_menos_markowitz"] = comparison_bl_vs_markowitz["retorno_anual_bl"] - comparison_bl_vs_markowitz["retorno_anual_markowitz"]
    comparison_bl_vs_markowitz["delta_drawdown_bl_menos_markowitz"] = comparison_bl_vs_markowitz["max_drawdown_bl"] - comparison_bl_vs_markowitz["max_drawdown_markowitz"]

    comp_path = OUTPUT_DIR / f"comparison_bl_vs_markowitz_h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos.csv"
    comparison_bl_vs_markowitz.to_csv(comp_path, index=False)
    display(comparison_bl_vs_markowitz.sort_values("delta_sharpe_bl_menos_markowitz", ascending=False).head(12))
    print(f"[OK] {comp_path.name}")

    marketcap_profiles = bl_rebalance_df[
        bl_rebalance_df["view_type"].isin(MARKETCAP_MOMENTUM_VIEW_TYPES)
        & (bl_rebalance_df["portfolio_type"] == "Perfil")
    ].copy()
    benchmark_cols = [
        "scenario",
        "view_type",
        "retorno_total_rebalanceado",
        "retorno_anual_rebalanceado",
        "volatilidad_anual_rebalanceada",
        "max_drawdown_rebalanceado",
        "sharpe_rebalanceado",
    ]
    marketcap_benchmarks = bl_rebalance_df[
        bl_rebalance_df["view_type"].isin(MARKETCAP_MOMENTUM_VIEW_TYPES)
        & (bl_rebalance_df["portfolio"] == "Benchmark")
    ][benchmark_cols].rename(
        columns={
            "retorno_total_rebalanceado": "benchmark_retorno_total",
            "retorno_anual_rebalanceado": "benchmark_retorno_anual",
            "volatilidad_anual_rebalanceada": "benchmark_volatilidad_anual",
            "max_drawdown_rebalanceado": "benchmark_max_drawdown",
            "sharpe_rebalanceado": "benchmark_sharpe",
        }
    )
    comparison_marketcap_momentum_profiles = marketcap_profiles.merge(
        marketcap_benchmarks,
        on=["scenario", "view_type"],
        how="left",
    )
    comparison_marketcap_momentum_profiles["view_label"] = comparison_marketcap_momentum_profiles["view_type"].map(BL_VIEW_LABELS)
    comparison_marketcap_momentum_profiles["delta_retorno_total_vs_benchmark"] = comparison_marketcap_momentum_profiles["retorno_total_rebalanceado"] - comparison_marketcap_momentum_profiles["benchmark_retorno_total"]
    comparison_marketcap_momentum_profiles["delta_retorno_anual_vs_benchmark"] = comparison_marketcap_momentum_profiles["retorno_anual_rebalanceado"] - comparison_marketcap_momentum_profiles["benchmark_retorno_anual"]
    comparison_marketcap_momentum_profiles["delta_volatilidad_anual_vs_benchmark"] = comparison_marketcap_momentum_profiles["volatilidad_anual_rebalanceada"] - comparison_marketcap_momentum_profiles["benchmark_volatilidad_anual"]
    comparison_marketcap_momentum_profiles["delta_drawdown_vs_benchmark"] = comparison_marketcap_momentum_profiles["max_drawdown_rebalanceado"] - comparison_marketcap_momentum_profiles["benchmark_max_drawdown"]
    comparison_marketcap_momentum_profiles["delta_sharpe_vs_benchmark"] = comparison_marketcap_momentum_profiles["sharpe_rebalanceado"] - comparison_marketcap_momentum_profiles["benchmark_sharpe"]

    marketcap_comp_path = OUTPUT_DIR / f"comparison_marketcap_momentum_profiles_h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos.csv"
    comparison_marketcap_momentum_profiles.to_csv(marketcap_comp_path, index=False)
    display(
        comparison_marketcap_momentum_profiles.sort_values(
            ["scenario_label", "view_type", "sharpe_rebalanceado"],
            ascending=[True, True, False],
        )[
            [
                "scenario_label",
                "view_label",
                "portfolio",
                "retorno_total_rebalanceado",
                "volatilidad_anual_rebalanceada",
                "max_drawdown_rebalanceado",
                "sharpe_rebalanceado",
                "delta_retorno_total_vs_benchmark",
                "delta_volatilidad_anual_vs_benchmark",
                "delta_drawdown_vs_benchmark",
                "delta_sharpe_vs_benchmark",
            ]
        ]
    )
    print(f"[OK] {marketcap_comp_path.name}")


,scenario,scenario_label,view_type,portfolio,portfolio_type,retorno_total_bl,retorno_anual_bl,volatilidad_anual_bl,max_drawdown_bl,sharpe_bl,retorno_total_markowitz,retorno_anual_markowitz,volatilidad_anual_markowitz,max_drawdown_markowitz,sharpe_markowitz,delta_sharpe_bl_menos_markowitz,delta_retorno_total_bl_menos_markowitz,delta_retorno_anual_bl_menos_markowitz,delta_drawdown_bl_menos_markowitz
48,con_pandemia,Con pandemia,desempleo,Arriesgado,Perfil,0.718662,0.310977,0.123541,-0.133222,2.355318,0.584212,0.258655,0.204574,-0.210520,1.166596,1.188721,0.134450,0.052323,0.077298
58,con_pandemia,Con pandemia,momentum_top20_6m,Muy arriesgado,Perfil,1.575058,0.604699,0.354350,-0.336520,1.650061,0.440470,0.200196,0.328408,-0.310421,0.548694,1.101367,1.134589,0.404503,-0.026099
49,con_pandemia,Con pandemia,desempleo,Muy arriesgado,Perfil,0.859685,0.363703,0.242626,-0.258073,1.416594,0.440470,0.200196,0.328408,-0.310421,0.548694,0.867899,0.419215,0.163507,0.052348
59,con_pandemia,Con pandemia,momentum_top20_6m,b_teorico,Especial,1.070873,0.439053,0.241293,-0.239009,1.736696,0.567015,0.251805,0.239889,-0.225198,0.966302,0.770394,0.503858,0.187248,-0.013812
57,con_pandemia,Con pandemia,momentum_top20_6m,Arriesgado,Perfil,1.027596,0.423937,0.210856,-0.200010,1.915702,0.584212,0.258655,0.204574,-0.210520,1.166596,0.749106,0.443384,0.165282,0.010511
68,con_pandemia,Con pandemia,momentum_top20_bottom20_1y,b_teorico,Especial,0.922602,0.386579,0.226352,-0.242449,1.619513,0.567015,0.251805,0.239889,-0.225198,0.966302,0.653211,0.355587,0.134775,-0.017251
50,con_pandemia,Con pandemia,desempleo,b_teorico,Especial,0.756054,0.325162,0.190258,-0.206165,1.603934,0.567015,0.251805,0.239889,-0.225198,0.966302,0.637632,0.189040,0.073357,0.019033
56,con_pandemia,Con pandemia,momentum_top20_6m,Neutro,Perfil,0.694848,0.301863,0.118625,-0.116782,2.376090,0.604652,0.266748,0.140526,-0.152040,1.755897,0.620193,0.090196,0.035115,0.035259
40,con_pandemia,Con pandemia,momentum,Muy arriesgado,Perfil,1.248944,0.499648,0.420640,-0.413586,1.140281,0.440470,0.200196,0.328408,-0.310421,0.548694,0.591586,0.808474,0.299452,-0.103164
66,con_pandemia,Con pandemia,momentum_top20_bottom20_1y,Arriesgado,Perfil,0.899372,0.378177,0.208773,-0.227313,1.715628,0.584212,0.258655,0.204574,-0.210520,1.166596,0.549031,0.315161,0.119522,-0.016793


[OK] comparison_bl_vs_markowitz_h7_f2_601activos.csv


,scenario_label,view_label,portfolio,retorno_total_rebalanceado,volatilidad_anual_rebalanceada,max_drawdown_rebalanceado,sharpe_rebalanceado,delta_retorno_total_vs_benchmark,delta_volatilidad_anual_vs_benchmark,delta_drawdown_vs_benchmark,delta_sharpe_vs_benchmark
12,Con pandemia,BL Momentum Top20 6M,Neutro,0.694848,0.118625,-0.116782,2.376090,-0.319459,-0.075932,0.102289,0.323926
11,Con pandemia,BL Momentum Top20 6M,Conservador,0.563805,0.109446,-0.097396,2.106264,-0.450502,-0.085111,0.121674,0.054100
10,Con pandemia,BL Momentum Top20 6M,Muy conservador,0.542628,0.109925,-0.095956,2.019796,-0.471678,-0.084632,0.123115,-0.032368
13,Con pandemia,BL Momentum Top20 6M,Arriesgado,1.027596,0.210856,-0.200010,1.915702,0.013289,0.016299,0.019061,-0.136462
14,Con pandemia,BL Momentum Top20 6M,Muy arriesgado,1.575058,0.354350,-0.336520,1.650061,0.560752,0.159793,-0.117449,-0.402103
17,Con pandemia,BL Momentum Top20-Bottom20 1Y,Neutro,0.656337,0.121418,-0.127863,2.198921,-0.357969,-0.073139,0.091208,0.146757
16,Con pandemia,BL Momentum Top20-Bottom20 1Y,Conservador,0.572766,0.110619,-0.099788,2.116267,-0.441540,-0.083938,0.119282,0.064103
15,Con pandemia,BL Momentum Top20-Bottom20 1Y,Muy conservador,0.544325,0.110290,-0.096779,2.019300,-0.469981,-0.084267,0.122291,-0.032864
18,Con pandemia,BL Momentum Top20-Bottom20 1Y,Arriesgado,0.899372,0.208773,-0.227313,1.715628,-0.114934,0.014216,-0.008243,-0.336536
19,Con pandemia,BL Momentum Top20-Bottom20 1Y,Muy arriesgado,0.860273,0.320739,-0.303854,1.072267,-0.154033,0.126183,-0.084784,-0.979897


[OK] comparison_marketcap_momentum_profiles_h7_f2_601activos.csv


### Resultado 3 - Black-Litterman vs Markowitz y Benchmark

Black-Litterman se compara contra el CSV final de Markowitz usando las mismas columnas de desempeno rebalanceado. La tabla muestra las mayores mejoras de Sharpe.

| Escenario | View | Portafolio | Sharpe BL | Sharpe Markowitz | Delta Sharpe | Delta retorno |
| --- | --- | --- | --- | --- | --- | --- |
| Con pandemia | desempleo | Arriesgado | 2.36 | 1.17 | 1.19 | 13.4% |
| Con pandemia | desempleo | Muy arriesgado | 1.42 | 0.55 | 0.87 | 41.9% |
| Con pandemia | desempleo | b_teorico | 1.60 | 0.97 | 0.64 | 18.9% |
| Con pandemia | momentum | Muy arriesgado | 1.14 | 0.55 | 0.59 | 80.8% |
| Con pandemia | momentum | b_teorico | 1.49 | 0.97 | 0.53 | 34.3% |
| Sin pandemia | desempleo | Arriesgado | 2.23 | 1.73 | 0.50 | -16.6% |
| Con pandemia | desempleo | Neutro | 2.13 | 1.76 | 0.38 | -4.0% |
| Sin pandemia | desempleo | Muy arriesgado | 1.60 | 1.41 | 0.19 | 0.9% |
| Sin pandemia | desempleo | b_teorico | 1.69 | 1.51 | 0.18 | -23.6% |
| Sin pandemia | momentum | Muy conservador | 2.38 | 2.21 | 0.17 | 6.0% |

![Ranking Sharpe BL](outputs/bl_ranking_sharpe_comparison.png)

**Lectura del resultado:** el benchmark top-20 equiponderado no cambia con las views; su delta Sharpe promedio contra Markowitz es **0.0000**. Por eso la comparacion relevante es BL vs Markowitz en los portafolios optimizados, especialmente donde las views corrigen el retorno esperado.


## Visualizaciones centrales

Los graficos resumen ranking de Sharpe, sensibilidad a pandemia y frontera eficiente BL. Sirven para leer rapidamente si BL mejora o no el desempeno de Markowitz y si la view elegida cambia la conclusion.


In [10]:
if RUN_CENTRAL_ANALYSIS:
    plot_df = bl_rebalance_df.copy()
    plot_df["label"] = plot_df["scenario_label"] + " | " + plot_df["view_type"] + " | " + plot_df["portfolio"]
    top_plot = plot_df.sort_values("sharpe_rebalanceado", ascending=False).head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(top_plot["label"], top_plot["sharpe_rebalanceado"], color="#2a9d8f")
    ax.set_title("Black-Litterman: ranking Sharpe rebalanceado")
    ax.set_xlabel("Sharpe rebalanceado")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "bl_ranking_sharpe_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 6.5))
    pivot = bl_rebalance_df.pivot_table(index=["view_type", "portfolio"], columns="scenario_label", values="sharpe_rebalanceado", aggfunc="first")
    sort_col = "Sin pandemia" if "Sin pandemia" in pivot.columns else pivot.columns[0]
    pivot = pivot.sort_values(sort_col, ascending=False).head(14)
    x = np.arange(len(pivot))
    width = 0.38
    labels = [f"{v}\n{p}" for v, p in pivot.index]
    ax.bar(x - width / 2, pivot.get("Sin pandemia", pd.Series(index=pivot.index, dtype=float)), width, label="Sin pandemia", color="#264653")
    ax.bar(x + width / 2, pivot.get("Con pandemia", pd.Series(index=pivot.index, dtype=float)), width, label="Con pandemia", color="#e76f51")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Sharpe rebalanceado")
    ax.set_title("BL: efecto de incluir pandemia")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "comparison_bl_sin_vs_con_pandemia.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    frontier_plot = bl_frontier_df.copy()
    view_types = [view for view in BL_VIEW_TYPES if view in set(frontier_plot["view_type"])]
    fig, axes = plt.subplots(1, len(view_types), figsize=(6.4 * len(view_types), 5.5), sharey=True, squeeze=False)
    for ax, view_type in zip(axes.ravel(), view_types):
        subset = frontier_plot[(frontier_plot["scenario"] == "sin_pandemia") & (frontier_plot["view_type"] == view_type)]
        for segment_id, group in subset.groupby("segment_id"):
            group = group.sort_values("volatilidad_anual_teorica")
            ax.plot(group["volatilidad_anual_teorica"], group["retorno_anual_teorico"], alpha=0.75, label=f"seg {segment_id}")
        ax.set_title(f"Frontera BL {view_type} - sin pandemia")
        ax.set_xlabel("Volatilidad anual teorica")
        ax.grid(alpha=0.25)
        ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
        ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    axes.ravel()[0].set_ylabel("Retorno anual teorico")
    axes.ravel()[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "bl_frontier_segments_h7_f2.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    print("[OK] PNG centrales guardados")


[OK] PNG centrales guardados


### Resultado 4 - Frontera eficiente Black-Litterman

La frontera se recalcula por segmento de rebalanceo. Esto permite ver si el posterior BL desplaza el set eficiente en cada periodo de calibracion.

![Frontera BL](outputs/bl_frontier_segments_h7_f2.png)

**Lectura del resultado:** cada linea es una frontera recalibrada. Las diferencias entre segmentos muestran que el modelo no usa una unica matriz estatica; reestima retorno esperado y covarianza con la informacion disponible al inicio de cada semestre.


## Validacion completa: 70 splits aleatorios

Se reutiliza exactamente `validation_windows.csv` del experimento de splits aleatorios. Para mantener la comparabilidad temporal, esta validacion se ejecuta sobre la base sin pandemia y procesa las mismas 70 ventanas. En cada ventana se evalua un subconjunto representativo: `Neutro`, `Minima varianza` y `Benchmark`.


In [11]:
def evaluate_bl_window(dataset: ScenarioDataset, window_row: pd.Series, view_type: str, validation_name: str) -> List[Dict[str, object]]:
    train, test = infer_window_by_dates(dataset.returns, window_row)
    _, _, view_record, results, _ = solve_bl_snapshot(
        train,
        test,
        dataset.tickers,
        dataset.meta,
        view_type,
        include_full_set=False,
    )
    rows = []
    for portfolio_type, result in results:
        if result.name not in VALIDATION_PORTFOLIOS:
            continue
        record = make_summary_record(
            str(window_row["split"]),
            int(window_row["historical_years"]),
            int(window_row["future_years"]),
            portfolio_type,
            result,
            dataset.tickers,
            train,
            test,
            DEFAULT_RISK_FREE_RATE,
            DEFAULT_INITIAL_CAPITAL,
        )
        record.update(
            {
                "validation": validation_name,
                "view_type": view_type,
                "horizon": window_row.get("horizon", f"h{int(window_row['historical_years'])}_f{int(window_row['future_years'])}"),
                "window_id": int(window_row.get("window_id", 1)),
                "window_start_idx": int(window_row["window_start_idx"]),
                "q_daily": view_record["q_daily"],
                "omega": view_record["omega"],
                "delta": view_record["delta"],
            }
        )
        rows.append(record)
    return rows


if RUN_VALIDATION_ANALYSIS:
    validation_dataset = load_scenario_dataset("sin_pandemia", MARKET_SIZE)
    random_windows = pd.read_csv(RANDOM_SPLITS_DIR / "validation_windows.csv")
    random_rows = []
    t0 = time.perf_counter()
    for i, row in random_windows.iterrows():
        if (i + 1) % 10 == 0:
            print(f"[RANDOM] ventana {i + 1}/{len(random_windows)}")
        for view_type in BL_VIEW_TYPES:
            random_rows.extend(evaluate_bl_window(validation_dataset, row, view_type, "random_splits"))

    bl_random_splits_summary = pd.DataFrame(random_rows)
    bl_random_splits_summary.to_csv(OUTPUT_DIR / "bl_random_splits_summary.csv", index=False)

    bl_random_splits_heatmap = (
        bl_random_splits_summary[bl_random_splits_summary["portfolio"] == "Neutro"]
        .groupby(["view_type", "historical_years", "future_years"], as_index=False)
        .agg(
            sharpe_real_futuro_promedio=("sharpe_real_futuro", "mean"),
            retorno_total_futuro_promedio=("retorno_total_futuro", "mean"),
            volatilidad_anual_real_futuro_promedio=("volatilidad_anual_real_futuro", "mean"),
            ventanas=("split", "nunique"),
        )
    )
    bl_random_splits_heatmap.to_csv(OUTPUT_DIR / "bl_random_splits_heatmap.csv", index=False)
    print(f"[OK] random splits: {len(bl_random_splits_summary)} filas en {time.perf_counter() - t0:.1f}s")
    display(bl_random_splits_heatmap.head())


## Heatmap train/test

El heatmap reporta el Sharpe futuro promedio del portafolio `Neutro` por combinacion train/test. La misma grilla permite comparar si la view de momentum o la view macro asumida cambian el split optimo.


In [12]:
if RUN_VALIDATION_ANALYSIS:
    view_types = [view for view in BL_VIEW_TYPES if view in set(bl_random_splits_heatmap["view_type"])]
    fig, axes = plt.subplots(1, len(view_types), figsize=(7 * len(view_types), 5.8), sharey=True, squeeze=False)
    for ax, view_type in zip(axes.ravel(), view_types):
        pivot = (
            bl_random_splits_heatmap[bl_random_splits_heatmap["view_type"] == view_type]
            .pivot(index="historical_years", columns="future_years", values="sharpe_real_futuro_promedio")
            .sort_index(ascending=False)
        )
        im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
        ax.set_title(f"Sharpe futuro promedio - {view_type}")
        ax.set_xlabel("Anios futuros de test")
        ax.set_ylabel("Anios historicos train")
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for y in range(pivot.shape[0]):
            for x in range(pivot.shape[1]):
                val = pivot.iloc[y, x]
                if np.isfinite(val):
                    ax.text(x, y, f"{val:.2f}", ha="center", va="center", color="white" if val < np.nanmean(pivot.values) else "black", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "bl_random_splits_heatmap.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("[OK] bl_random_splits_heatmap.png")


### Resultado 5 - Splits aleatorios: sensibilidad train/test

Se procesaron las mismas 70 ventanas aleatorias del experimento Markowitz. El heatmap usa el portafolio `Neutro` como portafolio representativo para comparar horizontes.

| View | Split optimo | Sharpe prom. | Retorno prom. | Vol prom. | Ventanas |
| --- | --- | --- | --- | --- | --- |
| desempleo | h6_f2 | 2.48 | 61.5% | 10.4% | 3 |
| momentum | h6_f2 | 2.31 | 68.6% | 13.1% | 3 |

![Heatmap splits BL](outputs/bl_random_splits_heatmap.png)

**Lectura del resultado:** en esta ejecucion, ambas variantes BL seleccionan `h6_f2` como mejor split aleatorio. Esto no invalida `h7_f2`; indica que el maximo promedio en la grilla aleatoria aparece con 6 anos de train y 2 de test, mientras que la validacion robusta oficial se revisa por separado.


## Validacion robusta: 18 ventanas moviles oficiales

Se reutilizan las 18 ventanas oficiales del desarrollo anterior. La tabla final agrupa por horizonte (`h7_f2` a `h2_f7`), view y portafolio, entregando promedio de Sharpe, retorno, volatilidad y drawdown.


In [13]:
if RUN_VALIDATION_ANALYSIS:
    robust_windows = pd.read_csv(ROBUST_WINDOWS_DIR / "validation_windows.csv")
    robust_rows = []
    t0 = time.perf_counter()
    for i, row in robust_windows.iterrows():
        print(f"[ROBUST] ventana {i + 1}/{len(robust_windows)}: {row['split']}")
        for view_type in BL_VIEW_TYPES:
            robust_rows.extend(evaluate_bl_window(validation_dataset, row, view_type, "robust_windows"))

    bl_robust_raw = pd.DataFrame(robust_rows)
    bl_robust_validation_summary = (
        bl_robust_raw.groupby(["view_type", "horizon", "historical_years", "future_years", "portfolio"], as_index=False)
        .agg(
            ventanas=("split", "nunique"),
            sharpe_real_futuro_promedio=("sharpe_real_futuro", "mean"),
            retorno_total_futuro_promedio=("retorno_total_futuro", "mean"),
            volatilidad_anual_real_futuro_promedio=("volatilidad_anual_real_futuro", "mean"),
            max_drawdown_futuro_promedio=("max_drawdown_futuro", "mean"),
        )
    )
    bl_robust_validation_summary.to_csv(OUTPUT_DIR / "bl_robust_validation_summary.csv", index=False)
    bl_robust_raw.to_csv(OUTPUT_DIR / "bl_robust_validation_windows_detail.csv", index=False)
    print(f"[OK] robust validation: {len(bl_robust_raw)} filas en {time.perf_counter() - t0:.1f}s")
    display(bl_robust_validation_summary.sort_values("sharpe_real_futuro_promedio", ascending=False).head(12))


In [14]:
if RUN_VALIDATION_ANALYSIS:
    fig, ax = plt.subplots(figsize=(11, 6))
    plot = bl_robust_validation_summary[bl_robust_validation_summary["portfolio"] == "Neutro"].copy()
    for view_type, group in plot.groupby("view_type"):
        group = group.sort_values("future_years")
        ax.plot(group["horizon"], group["sharpe_real_futuro_promedio"], marker="o", linewidth=2, label=view_type)
    ax.set_title("Validacion robusta BL: Sharpe futuro promedio por horizonte")
    ax.set_xlabel("Horizonte train/test")
    ax.set_ylabel("Sharpe futuro promedio")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "bl_robust_validation_summary.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("[OK] bl_robust_validation_summary.png")


### Resultado 6 - Validacion robusta con 18 ventanas moviles

La validacion oficial usa 6 horizontes por 3 ventanas. La tabla resume el mejor horizonte para el portafolio `Neutro` por tipo de view.

| View | Horizonte | Sharpe prom. | Retorno prom. | Vol prom. | Drawdown prom. |
| --- | --- | --- | --- | --- | --- |
| desempleo | h7_f2 | 2.35 | 59.2% | 10.6% | -9.7% |
| momentum | h7_f2 | 2.23 | 68.5% | 13.6% | -12.8% |

![Validacion robusta BL](outputs/bl_robust_validation_summary.png)

**Lectura del resultado:** `h7_f2` queda como mejor horizonte robusto para las views evaluadas. Esto respalda la tesis original: mas historia estabiliza la estimacion, pero el test de 2 anos mantiene vigencia temporal.


## Checklist final de validacion

Esta celda comprueba existencia de archivos, conteos esperados del diseno experimental y ausencia de NaN criticos en metricas clave. Si falla, el notebook no debe considerarse cerrado.


In [15]:
if RUN_CENTRAL_ANALYSIS:
    output_label = f"h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos"
    expected_files = [
        f"bl_rebalance_summary_{output_label}.csv",
        f"bl_rebalance_segments_{output_label}.csv",
        f"bl_segment_summary_{output_label}.csv",
        f"bl_frontier_segments_{output_label}.csv",
        f"bl_views_{output_label}.csv",
        f"comparison_bl_sin_vs_con_pandemia_{output_label}.csv",
        f"comparison_bl_vs_markowitz_{output_label}.csv",
        f"comparison_marketcap_momentum_profiles_{output_label}.csv",
        "bl_ranking_sharpe_comparison.png",
        "comparison_bl_sin_vs_con_pandemia.png",
        "bl_frontier_segments_h7_f2.png",
    ]
    if RUN_VALIDATION_ANALYSIS:
        expected_files.extend([
            "bl_random_splits_summary.csv",
            "bl_random_splits_heatmap.csv",
            "bl_robust_validation_summary.csv",
            "bl_random_splits_heatmap.png",
            "bl_robust_validation_summary.png",
        ])
    missing_outputs = [name for name in expected_files if not (OUTPUT_DIR / name).exists()]
    if missing_outputs:
        raise FileNotFoundError("Faltan outputs BL:\n" + "\n".join(missing_outputs))

    checks = {
        "central_rebalance_rows": len(bl_rebalance_df),
        "central_segments_rows": len(bl_segments_df),
        "views_rows": len(bl_views_df),
        "views": ", ".join(sorted(bl_rebalance_df["view_type"].unique())),
    }
    expected_rebalance = len(SCENARIOS) * len(BL_VIEW_TYPES) * 9
    expected_views = len(SCENARIOS) * len(BL_VIEW_TYPES) * 4
    assert len(bl_rebalance_df) == expected_rebalance, checks
    assert len(bl_views_df) == expected_views, checks
    if RUN_VALIDATION_ANALYSIS:
        checks.update({
            "random_windows_processed": bl_random_splits_summary["split"].nunique(),
            "robust_windows_processed": bl_robust_raw["split"].nunique(),
            "random_rows": len(bl_random_splits_summary),
            "robust_rows": len(bl_robust_raw),
        })
        assert checks["random_windows_processed"] == 70, checks
        assert checks["robust_windows_processed"] == 18, checks
        assert checks["random_rows"] == 70 * len(BL_VIEW_TYPES) * len(VALIDATION_PORTFOLIOS), checks
        assert checks["robust_rows"] == 18 * len(BL_VIEW_TYPES) * len(VALIDATION_PORTFOLIOS), checks

    critical_frames = {
        "bl_rebalance_df": (bl_rebalance_df, ["sharpe_rebalanceado", "retorno_total_rebalanceado", "volatilidad_anual_rebalanceada"]),
    }
    if RUN_VALIDATION_ANALYSIS:
        critical_frames.update({
            "bl_random_splits_summary": (bl_random_splits_summary, ["sharpe_real_futuro", "retorno_total_futuro", "volatilidad_anual_real_futuro"]),
            "bl_robust_raw": (bl_robust_raw, ["sharpe_real_futuro", "retorno_total_futuro", "volatilidad_anual_real_futuro"]),
        })
    for name, (df, cols) in critical_frames.items():
        bad = df[cols].isna().sum()
        if int(bad.sum()) > 0:
            raise ValueError(f"NaN criticos en {name}: {bad.to_dict()}")

    checklist_df = pd.DataFrame([checks]).T.rename(columns={0: "valor"})
    display(checklist_df)
    print("[OK] Checklist final completo")


,valor
central_rebalance_rows,72
central_segments_rows,288
views_rows,32
views,"desempleo, momentum, momentum_top20_6m, moment..."


[OK] Checklist final completo


## Conclusiones dinamicas

Las conclusiones se generan desde los CSV calculados en este mismo notebook. Responden: split optimo, efecto pandemia, rebalanceo y comparacion Benchmark vs Markowitz vs Black-Litterman.


In [16]:
if RUN_CENTRAL_ANALYSIS:
    best_central = (
        bl_rebalance_df[bl_rebalance_df["portfolio"] != "Benchmark"]
        .sort_values("sharpe_rebalanceado", ascending=False)
        .groupby(["scenario_label", "view_type"], as_index=False)
        .head(1)
    )
    best_vs_markowitz = comparison_bl_vs_markowitz.sort_values("delta_sharpe_bl_menos_markowitz", ascending=False).head(1)
    benchmark_rows = comparison_bl_vs_markowitz[comparison_bl_vs_markowitz["portfolio"] == "Benchmark"].copy()
    benchmark_delta = benchmark_rows["delta_sharpe_bl_menos_markowitz"].dropna().mean()
    marketcap_winners = (
        comparison_marketcap_momentum_profiles.sort_values("sharpe_rebalanceado", ascending=False)
        .groupby("scenario_label", as_index=False)
        .head(1)
    )

    lines = ["# Conclusiones Black-Litterman\n"]
    if RUN_VALIDATION_ANALYSIS and "bl_random_splits_heatmap" in globals():
        best_random = (
            bl_random_splits_heatmap.sort_values("sharpe_real_futuro_promedio", ascending=False)
            .groupby("view_type", as_index=False)
            .head(1)
        )
        for _, row in best_random.iterrows():
            lines.append(
                f"- Split optimo aleatorio para BL {row['view_type']}: h{int(row['historical_years'])}_f{int(row['future_years'])}, "
                f"Sharpe promedio {row['sharpe_real_futuro_promedio']:.2f} en {int(row['ventanas'])} ventanas."
            )

    for _, row in best_central.iterrows():
        lines.append(
            f"- Experimento central {row['scenario_label']} / {row['view_type']}: lidera {row['portfolio']} "
            f"con Sharpe rebalanceado {row['sharpe_rebalanceado']:.2f}, retorno total {row['retorno_total_rebalanceado']:.1%} "
            f"y drawdown {row['max_drawdown_rebalanceado']:.1%}."
        )

    lines.append("\n## Comparativa especifica de momentum market-cap")
    for _, row in marketcap_winners.iterrows():
        lines.append(
            f"- En {row['scenario_label']}, entre las variantes market-cap gana {row['view_label']} con perfil {row['portfolio']}: "
            f"Sharpe {row['sharpe_rebalanceado']:.2f}, retorno total {row['retorno_total_rebalanceado']:.1%}, "
            f"volatilidad anual {row['volatilidad_anual_rebalanceada']:.1%} y drawdown {row['max_drawdown_rebalanceado']:.1%}. "
            f"Frente al benchmark cambia Sharpe en {row['delta_sharpe_vs_benchmark']:+.2f}, retorno total en {row['delta_retorno_total_vs_benchmark']:+.1%}, "
            f"volatilidad en {row['delta_volatilidad_anual_vs_benchmark']:+.1%} y drawdown en {row['delta_drawdown_vs_benchmark']:+.1%}."
        )

    lines.append(
        "- BL Momentum Top20 6M usa las 20 mayores capitalizaciones disponibles y separa 10 ganadoras contra 10 perdedoras por momentum de 126 dias; "
        "es la variante mas concentrada y reactiva."
    )
    lines.append(
        "- BL Momentum Top20-Bottom20 1Y usa las 40 mayores capitalizaciones disponibles y separa 20 ganadoras contra 20 perdedoras por momentum de 252 dias; "
        "es la variante mas diversificada y lenta."
    )

    if len(best_vs_markowitz):
        r = best_vs_markowitz.iloc[0]
        lines.append(
            f"- Comparacion contra Markowitz: la mayor mejora BL aparece en {r['scenario_label']} / {r['view_type']} / {r['portfolio']}, "
            f"con delta Sharpe {r['delta_sharpe_bl_menos_markowitz']:.2f}."
        )
    lines.append(
        f"- Benchmark: al ser equiponderado top-{BENCHMARK_SIZE}, no depende de las views BL; su delta Sharpe promedio contra el CSV Markowitz es {benchmark_delta:.4f}."
    )
    lines.append(
        "- Justificacion comparativa: el benchmark suele capturar mas retorno bruto cuando concentra riesgo en las megacaps, mientras que las variantes BL se justifican si elevan Sharpe o reducen drawdown; por eso la conclusion prioriza rendimiento ajustado por riesgo y no solo retorno total."
    )
    lines.append(
        "- Rebalanceo: todas las views BL se actualizan cada segmento semestral de 126 dias habiles y se reportan metricas agregadas del camino rebalanceado."
    )

    conclusion_text = "\n".join(lines)
    display(Markdown(conclusion_text))
    (OUTPUT_DIR / "conclusiones_black_litterman.md").write_text(conclusion_text, encoding="utf-8")


# Conclusiones Black-Litterman

- Experimento central Sin pandemia / momentum_top20_6m: lidera Neutro con Sharpe rebalanceado 2.46, retorno total 72.3% y drawdown -12.9%.
- Experimento central Sin pandemia / desempleo: lidera Neutro con Sharpe rebalanceado 2.44, retorno total 63.1% y drawdown -10.4%.
- Experimento central Sin pandemia / momentum: lidera Muy conservador con Sharpe rebalanceado 2.38, retorno total 64.0% y drawdown -10.1%.
- Experimento central Con pandemia / momentum_top20_6m: lidera Neutro con Sharpe rebalanceado 2.38, retorno total 69.5% y drawdown -11.7%.
- Experimento central Con pandemia / desempleo: lidera Arriesgado con Sharpe rebalanceado 2.36, retorno total 71.9% y drawdown -13.3%.
- Experimento central Sin pandemia / momentum_top20_bottom20_1y: lidera Conservador con Sharpe rebalanceado 2.35, retorno total 63.1% y drawdown -10.7%.
- Experimento central Con pandemia / momentum_top20_bottom20_1y: lidera Neutro con Sharpe rebalanceado 2.20, retorno total 65.6% y drawdown -12.8%.
- Experimento central Con pandemia / momentum: lidera Conservador con Sharpe rebalanceado 2.06, retorno total 56.4% y drawdown -10.3%.

## Comparativa especifica de momentum market-cap
- En Sin pandemia, entre las variantes market-cap gana BL Momentum Top20 6M con perfil Neutro: Sharpe 2.46, retorno total 72.3%, volatilidad anual 11.9% y drawdown -12.9%. Frente al benchmark cambia Sharpe en +0.41, retorno total en -29.2%, volatilidad en -7.6% y drawdown en +9.0%.
- En Con pandemia, entre las variantes market-cap gana BL Momentum Top20 6M con perfil Neutro: Sharpe 2.38, retorno total 69.5%, volatilidad anual 11.9% y drawdown -11.7%. Frente al benchmark cambia Sharpe en +0.32, retorno total en -31.9%, volatilidad en -7.6% y drawdown en +10.2%.
- BL Momentum Top20 6M usa las 20 mayores capitalizaciones disponibles y separa 10 ganadoras contra 10 perdedoras por momentum de 126 dias; es la variante mas concentrada y reactiva.
- BL Momentum Top20-Bottom20 1Y usa las 40 mayores capitalizaciones disponibles y separa 20 ganadoras contra 20 perdedoras por momentum de 252 dias; es la variante mas diversificada y lenta.
- Comparacion contra Markowitz: la mayor mejora BL aparece en Con pandemia / desempleo / Arriesgado, con delta Sharpe 1.19.
- Benchmark: al ser equiponderado top-20, no depende de las views BL; su delta Sharpe promedio contra el CSV Markowitz es 0.0000.
- Justificacion comparativa: el benchmark suele capturar mas retorno bruto cuando concentra riesgo en las megacaps, mientras que las variantes BL se justifican si elevan Sharpe o reducen drawdown; por eso la conclusion prioriza rendimiento ajustado por riesgo y no solo retorno total.
- Rebalanceo: todas las views BL se actualizan cada segmento semestral de 126 dias habiles y se reportan metricas agregadas del camino rebalanceado.

In [17]:
if RUN_CENTRAL_ANALYSIS:
    output_label = f"h{HISTORICAL_YEARS}_f{FUTURE_YEARS}_{MARKET_SIZE}activos"
    markowitz_path = MARKOWITZ_OUTPUT_DIR / f"rebalance_summary_{output_label}.csv"
    mk = pd.read_csv(markowitz_path)
    if "scenario" not in mk.columns:
        first = mk.columns[0]
        mk = mk.rename(columns={first: "scenario"})

    kpi_rows = []
    metric_cols = [
        "n_segments",
        "retorno_total_rebalanceado",
        "retorno_anual_rebalanceado",
        "volatilidad_anual_rebalanceada",
        "max_drawdown_rebalanceado",
        "cvar_95_diario_rebalanceado",
        "sharpe_rebalanceado",
    ]

    def append_kpi(row: pd.Series, modelo: str, view: str, criterio: str):
        out = {
            "scenario": row["scenario"],
            "scenario_label": row.get("scenario_label", row["scenario"]),
            "modelo": modelo,
            "view": view,
            "criterio_seleccion": criterio,
            "portfolio_seleccionado": row["portfolio"],
        }
        for col in metric_cols:
            out[col] = row[col]
        kpi_rows.append(out)

    for scenario in sorted(mk["scenario"].dropna().unique()):
        mk_s = mk[mk["scenario"] == scenario].copy()
        bl_s = bl_rebalance_df[bl_rebalance_df["scenario"] == scenario].copy()

        benchmark = mk_s[mk_s["portfolio"] == "Benchmark"].iloc[0]
        append_kpi(benchmark, "Equiponderado", "No aplica", "Benchmark top-20 equiponderado")

        markowitz_candidates = mk_s[mk_s["portfolio"] != "Benchmark"].copy()
        markowitz_best = markowitz_candidates.sort_values("sharpe_rebalanceado", ascending=False).iloc[0]
        append_kpi(markowitz_best, "Markowitz", "Retornos historicos", "Mejor Sharpe rebalanceado entre portafolios Markowitz")

        for view_type in BL_VIEW_TYPES:
            view_candidates = bl_s[(bl_s["view_type"] == view_type) & (bl_s["portfolio"] != "Benchmark")].copy()
            if view_candidates.empty:
                continue
            best_bl = view_candidates.sort_values("sharpe_rebalanceado", ascending=False).iloc[0]
            append_kpi(best_bl, "Black-Litterman", view_type, f"Mejor Sharpe rebalanceado BL {view_type}")

    comparison_three_models = pd.DataFrame(kpi_rows)
    comparison_three_models.to_csv(OUTPUT_DIR / f"comparison_three_models_kpis_{output_label}.csv", index=False)
    display(comparison_three_models.sort_values(["scenario_label", "sharpe_rebalanceado"], ascending=[True, False]))
    print(f"[OK] comparison_three_models_kpis_{output_label}.csv")


,scenario,scenario_label,modelo,view,criterio_seleccion,portfolio_seleccionado,n_segments,retorno_total_rebalanceado,retorno_anual_rebalanceado,volatilidad_anual_rebalanceada,max_drawdown_rebalanceado,cvar_95_diario_rebalanceado,sharpe_rebalanceado
4,con_pandemia,Con pandemia,Black-Litterman,momentum_top20_6m,Mejor Sharpe rebalanceado BL momentum_top20_6m,Neutro,4,0.694848,0.301863,0.118625,-0.116782,0.015805,2.376090
3,con_pandemia,Con pandemia,Black-Litterman,desempleo,Mejor Sharpe rebalanceado BL desempleo,Arriesgado,4,0.718662,0.310977,0.123541,-0.133222,0.017117,2.355318
5,con_pandemia,Con pandemia,Black-Litterman,momentum_top20_bottom20_1y,Mejor Sharpe rebalanceado BL momentum_top20_bo...,Neutro,4,0.656337,0.286988,0.121418,-0.127863,0.016654,2.198921
2,con_pandemia,Con pandemia,Black-Litterman,momentum,Mejor Sharpe rebalanceado BL momentum,Conservador,4,0.563570,0.250428,0.111616,-0.103267,0.014667,2.064471
0,con_pandemia,Con pandemia,Equiponderado,No aplica,Benchmark top-20 equiponderado,Benchmark,4,1.014306,0.419263,0.194557,-0.219071,0.027896,2.052164
1,con_pandemia,Con pandemia,Markowitz,Retornos historicos,Mejor Sharpe rebalanceado entre portafolios Ma...,Muy conservador,4,0.545485,0.243175,0.110312,-0.102117,0.014950,2.023131
7,sin_pandemia,Sin pandemia,Markowitz,Retornos historicos,Mejor Sharpe rebalanceado entre portafolios Ma...,Ideal de mercado,4,0.764785,0.328452,0.123031,-0.126262,0.016736,2.507113
10,sin_pandemia,Sin pandemia,Black-Litterman,momentum_top20_6m,Mejor Sharpe rebalanceado BL momentum_top20_6m,Neutro,4,0.722636,0.312492,0.118921,-0.128718,0.015886,2.459544
9,sin_pandemia,Sin pandemia,Black-Litterman,desempleo,Mejor Sharpe rebalanceado BL desempleo,Neutro,4,0.630909,0.277070,0.105416,-0.103851,0.014058,2.438621
8,sin_pandemia,Sin pandemia,Black-Litterman,momentum,Mejor Sharpe rebalanceado BL momentum,Muy conservador,4,0.640050,0.280644,0.109445,-0.101281,0.014506,2.381504


[OK] comparison_three_models_kpis_h7_f2_601activos.csv


## Conclusiones finales del diseno Black-Litterman

Las conclusiones y la tabla comparativa final se generan dinamicamente en las celdas anteriores y se guardan en `outputs/conclusiones_black_litterman.md` y `outputs/comparison_three_models_kpis_h7_f2_601activos.csv`.

La comparacion incluye Equiponderado, Markowitz, BL Momentum, BL Desempleo, `momentum_top20_6m` y `momentum_top20_bottom20_1y`. La comparativa especifica de perfiles market-cap momentum contra benchmark se guarda en `outputs/comparison_marketcap_momentum_profiles_h7_f2_601activos.csv`.
